# Pipeline Forecasting Harga Pangan Nasional (SEC SATRIA DATA 2026)

Notebook ini merupakan gabungan dari seluruh pipeline analisis untuk dijalankan secara **lokal**.
Output (CSV dan grafik) akan langsung disimpan ke dalam folder lokal proyek.

> **PENTING:** Dataset di-load langsung dari GitHub. Output disimpan ke folder lokal `outputs/` di direktori proyek.

In [ ]:
# 1. INSTALL LIBRARY YANG DIBUTUHKAN
%pip install prophet
%pip install statsmodels
%pip install matplotlib
%pip install seaborn

In [ ]:
# 2. SETUP FOLDER OUTPUT PORTABLE

import os
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# SETUP PATH PORTABLE
# ============================================================================
def _resolve_project_dir():
    """Ambil root proyek; kalau dijalankan dari folder Code, naik ke folder induk."""
    env_dir = os.environ.get("PROJECT_DIR")
    if env_dir:
        return os.path.abspath(env_dir)

    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()

    base_dir = os.path.abspath(base_dir)
    parent_dir = os.path.abspath(os.path.join(base_dir, os.pardir))

    # Struktur proyek: <PROJECT_DIR>/Code/code.ipynb, <PROJECT_DIR>/dataset, <PROJECT_DIR>/outputs
    # Jadi saat notebook/script dijalankan dari folder Code, output tetap ke folder outputs di root proyek.
    if os.path.basename(base_dir).lower() == "code":
        return parent_dir

    if os.path.isdir(os.path.join(base_dir, "dataset")):
        return base_dir
    if os.path.isdir(os.path.join(parent_dir, "dataset")):
        return parent_dir

    return base_dir

PROJECT_DIR = _resolve_project_dir()
OUTPUT_DIR  = os.path.join(PROJECT_DIR, "outputs")
FIG_DIR     = os.path.join(OUTPUT_DIR, "figures")
TAB_DIR     = os.path.join(OUTPUT_DIR, "tables")
OUTPUT_DATA = OUTPUT_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

dataaa_dir  = os.path.join(PROJECT_DIR, 'dataaa')

print(f"Direktori proyek: {PROJECT_DIR}")
print(f"Direktori output: {OUTPUT_DIR}")


## Tahap: 00_data_enrichment.py

In [ ]:
# -*- coding: utf-8 -*-
"""
PHASE 0 (FIXED): DATA ENRICHMENT  -  Pipeline Rekonstruksi Aktual
================================================================
Perbaikan utama:
  - COMMODITY_VARIANTS: mapping nama target ke seluruh varian nama WFP
  - Rekonstruksi diperluas ke semua bulan yang punya data pasar (tidak dibatasi Mei 2024)
  - Kalibrasi rasio dihitung ulang dari overlap Feb-Mar 2020
  - Komoditas Fuel (kerosene), Milk (condensed), Wheat flour dihapus dari target
  - Cabai: rekonstruksi aktual hingga Mei 2024, seasonal estimate sisanya

Input  : dataset/wfp_food_prices_idn.csv
Output : outputs/national_avg_clean_fixed.csv
"""

import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# ===========================================================================
# KONFIGURASI PATH PORTABLE
# ===========================================================================
def _resolve_project_dir():
    """Ambil root proyek; kalau dijalankan dari folder Code, naik ke folder induk."""
    env_dir = os.environ.get("PROJECT_DIR")
    if env_dir:
        return os.path.abspath(env_dir)

    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()

    base_dir = os.path.abspath(base_dir)
    parent_dir = os.path.abspath(os.path.join(base_dir, os.pardir))

    # Struktur proyek: <PROJECT_DIR>/Code/code.ipynb, <PROJECT_DIR>/dataset, <PROJECT_DIR>/outputs
    # Jadi saat notebook/script dijalankan dari folder Code, output tetap ke folder outputs di root proyek.
    if os.path.basename(base_dir).lower() == "code":
        return parent_dir

    if os.path.isdir(os.path.join(base_dir, "dataset")):
        return base_dir
    if os.path.isdir(os.path.join(parent_dir, "dataset")):
        return parent_dir

    return base_dir

PROJECT_DIR = _resolve_project_dir()
INPUT_FILE  = os.path.join(PROJECT_DIR, 'dataset', 'wfp_food_prices_idn.csv')
OUTPUT_DIR  = os.path.join(PROJECT_DIR, 'outputs')
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'national_avg_clean_fixed.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_COMMODITIES = [
    'Rice', 'Eggs', 'Meat (beef)', 'Meat (chicken, broiler)',
    'Sugar', 'Oil (vegetable)',
    "Chili (bird's eye)", 'Chili (red)'
]

# Mapping: nama target -> semua varian nama komoditas di WFP yang relevan
# Ini kunci perbaikan: data 2024-2026 memakai nama baru seperti
# 'Rice (medium quality)', 'Meat (beef, first quality)', dll.
COMMODITY_VARIANTS = {
    'Rice'                   : ['Rice', 'Rice (medium quality)', 'Rice (high quality)', 'Rice (low quality)'],
    'Eggs'                   : ['Eggs', 'Eggs (broiler)'],
    'Meat (beef)'            : ['Meat (beef)', 'Meat (beef, first quality)', 'Meat (beef, second quality)'],
    'Meat (chicken, broiler)': ['Meat (chicken, broiler)', 'Meat (chicken)'],
    'Sugar'                  : ['Sugar', 'Sugar (local)', 'Sugar (premium)'],
    'Oil (vegetable)'        : ['Oil (vegetable)', 'Oil (vegetable, bulk)', 'Oil (vegetable, packaged)'],
    "Chili (bird's eye)"     : ["Chili (bird's eye)", "Chili (bird's eye, red)", "Chili (bird's eye, green)"],
    'Chili (red)'            : ['Chili (red)', 'Chili (red, curly)', 'Chili (red, large)'],
}

# Komoditas yang data pasar regionalnya tersedia (bisa direkonstruksi)
RECONSTRUCTABLE = [
    'Rice', 'Eggs', 'Meat (beef)', 'Meat (chicken, broiler)',
    'Sugar', 'Oil (vegetable)', "Chili (bird's eye)", 'Chili (red)'
]

# Komoditas yang masih perlu estimasi untuk Jun 2024-Jan 2026
# setelah data pasar regional habis
NEEDS_ESTIMATION = [
    "Chili (bird's eye)",
    'Chili (red)',
]

SEASONAL_COMMODITIES = ["Chili (bird's eye)", "Chili (red)"]

# Rasio kalibrasi dihitung ulang dari overlap Feb-Mar 2020
CALIBRATION_RATIOS = {
    'Rice'                   : 1.222994,
    'Eggs'                   : 0.970739,
    'Meat (beef)'            : 0.937070,
    'Meat (chicken, broiler)': 1.354210,
    'Sugar'                  : 0.874149,
    'Oil (vegetable)'        : 1.117764,
    "Chili (bird's eye)"     : 1.453646,
    'Chili (red)'            : 1.455423,
}

COMMODITY_META = {
    'Rice'                  : {'id': 52,  'category': 'cereals and tubers',    'unit': 'KG'},
    'Eggs'                  : {'id': 92,  'category': 'meat, fish and eggs',   'unit': 'KG'},
    'Meat (beef)'           : {'id': 141, 'category': 'meat, fish and eggs',   'unit': 'KG'},
    'Meat (chicken, broiler)':{'id': 89,  'category': 'meat, fish and eggs',   'unit': 'KG'},
    'Sugar'                 : {'id': 97,  'category': 'miscellaneous food',    'unit': 'KG'},
    'Oil (vegetable)'       : {'id': 96,  'category': 'oil and fats',          'unit': 'L'},
    "Chili (bird's eye)"    : {'id': 604, 'category': 'vegetables and fruits', 'unit': 'KG'},
    'Chili (red)'           : {'id': 91,  'category': 'vegetables and fruits', 'unit': 'KG'},
}

TARGET_END = pd.Timestamp('2026-01-15')

print("=" * 70)
print("PHASE 0 (FIXED): DATA ENRICHMENT  -  Rekonstruksi Aktual WFP")
print("=" * 70)

# ===========================================================================
# 1. LOAD DATA ASLI
# ===========================================================================
print("\n[1] Loading dataset...")
df_raw = pd.read_csv(INPUT_FILE, parse_dates=['date'])
print(f"    Total baris    : {len(df_raw):,}")
print(f"    Rentang tanggal: {df_raw['date'].min().date()} -> {df_raw['date'].max().date()}")
print(f"    Jumlah market  : {df_raw['market'].nunique()}")

# ===========================================================================
# 2. EKSTRAK DATA NATIONAL AVERAGE (seluruh periode yang tersedia)
# ===========================================================================
print("\n[2] Ekstrak National Average asli dari WFP...")

df_nat_raw = df_raw[df_raw['market'] == 'National Average'].copy()

# Normalisasi: map semua varian nama ke nama target
nat_rows = []
for target, variants in COMMODITY_VARIANTS.items():
    subset = df_nat_raw[df_nat_raw['commodity'].isin(variants + [target])].copy()
    if len(subset) == 0:
        continue
    # Jika ada beberapa varian per tanggal, ambil rata-rata
    grouped = subset.groupby('date').agg(
        price=('price', 'mean'),
        usdprice=('usdprice', 'mean')
    ).reset_index()
    grouped['commodity'] = target
    grouped['source'] = 'WFP_National_Average'
    nat_rows.append(grouped)

df_nat = pd.concat(nat_rows, ignore_index=True)
print(f"    Baris diekstrak: {len(df_nat):,}")
print(f"    Komoditas      : {df_nat['commodity'].nunique()}")
print(f"    Rentang        : {df_nat['date'].min().date()} -> {df_nat['date'].max().date()}")
print()
# Ringkasan per komoditas
cov = df_nat.groupby('commodity').agg(
    first=('date', 'min'), last=('date', 'max'), rows=('price', 'count')
)
print(cov.to_string())

# ===========================================================================
# 3. REKONSTRUKSI NATIONAL AVERAGE DARI DATA PASAR (semua periode tersedia)
#    PERBAIKAN: tidak dibatasi tanggal, dan menggunakan COMMODITY_VARIANTS
# ===========================================================================
print("\n[3] Rekonstruksi National Average dari semua data pasar yang tersedia...")

df_pasar_raw = df_raw[df_raw['market'] != 'National Average'].copy()

# Rekonstruksi per komoditas target
recon_rows = []

for target in RECONSTRUCTABLE:
    variants = COMMODITY_VARIANTS.get(target, [target])
    subset = df_pasar_raw[df_pasar_raw['commodity'].isin(variants + [target])].copy()

    if len(subset) == 0:
        print(f"    [{target}] Tidak ada data pasar -> skip")
        continue

    # Rata-rata lintas pasar dan varian per bulan
    monthly = subset.groupby('date').agg(
        price_mean=('price', 'mean'),
        usdprice_mean=('usdprice', 'mean'),
        n_markets=('market', 'nunique')
    ).reset_index()

    # Terapkan kalibrasi
    ratio = CALIBRATION_RATIOS.get(target, 1.0)
    monthly['price']    = monthly['price_mean'] * ratio
    monthly['usdprice'] = monthly['usdprice_mean'] * ratio
    monthly['commodity'] = target
    monthly['source'] = monthly['n_markets'].apply(
        lambda n: f'reconstructed_avg_{n}markets'
    )

    recon_rows.append(monthly[['date', 'commodity', 'price', 'usdprice', 'source']])

    last = monthly.iloc[-1]
    print(f"    [{target}] {len(monthly)} bulan | "
          f"terakhir: {last['date'].date()} | "
          f"n_pasar: {int(last['n_markets'])} | "
          f"harga: Rp {last['price']:,.0f}")

df_reconstructed = pd.concat(recon_rows, ignore_index=True)
print(f"\n    Total baris direkonstruksi: {len(df_reconstructed):,}")

# Hapus bulan yang sudah ada di National Average (prioritaskan WFP asli)
existing_pairs = set(zip(df_nat['date'], df_nat['commodity']))
mask_new = ~df_reconstructed.apply(
    lambda r: (r['date'], r['commodity']) in existing_pairs, axis=1
)
df_reconstructed = df_reconstructed[mask_new].copy()
print(f"    Setelah deduplikasi (bulan baru saja): {len(df_reconstructed):,}")

# ===========================================================================
# 4. ESTIMASI UNTUK KOMODITAS YANG TIDAK ADA DATA PASAR
#    Hanya: Wheat flour, Milk (condensed), Fuel (kerosene),
#    dan cabai untuk periode Jun 2024 - Jan 2026
# ===========================================================================
print("\n[4] Estimasi untuk komoditas tanpa data pasar aktual...")

estimated_rows = []

for commodity in NEEDS_ESTIMATION:
    hist_nat = df_nat[df_nat['commodity'] == commodity].copy()
    hist_rec = df_reconstructed[df_reconstructed['commodity'] == commodity].copy()

    hist = (
        pd.concat([hist_nat, hist_rec], ignore_index=True)
        .sort_values('date')
        .drop_duplicates(subset=['date'])
    )

    if len(hist) < 12:
        print(f"    [{commodity}] Data terlalu sedikit -> skip")
        continue

    last_date = hist['date'].max()

    if last_date >= TARGET_END:
        print(f"    [{commodity}] Sudah lengkap hingga {last_date.date()} -> skip")
        continue

    future_months = pd.date_range(
        start=last_date + pd.DateOffset(months=1),
        end=TARGET_END,
        freq='MS'
    ) + pd.DateOffset(days=14)

    n_months = len(future_months)

    if commodity in SEASONAL_COMMODITIES:
        # Seasonal estimation untuk cabai
        recent = hist.tail(24).copy()
        monthly_pattern = (
            recent.groupby(recent['date'].dt.month)['price'].mean()
        )
        base_price = recent['price'].mean()
        base_usd   = recent['usdprice'].mean()

        for dt in future_months:
            month_factor = monthly_pattern.get(dt.month, base_price) / base_price
            est_price    = base_price * month_factor
            estimated_rows.append({
                'date'     : dt,
                'commodity': commodity,
                'price'    : round(est_price, 2),
                'usdprice' : round(base_usd * month_factor, 6),
                'source'   : 'estimated_seasonal'
            })
        print(f"    [{commodity}] Seasonal estimation ({n_months} bulan: "
              f"{future_months[0].date()} -> {future_months[-1].date()})")

    else:
        # Fallback umum bila suatu komoditas non-seasonal ditambahkan di masa depan
        recent = hist.tail(24)
        n = len(recent)
        x = np.arange(n)

        slope_p, _ = np.polyfit(x, recent['price'].values, 1)
        slope_u, _ = np.polyfit(x, recent['usdprice'].values, 1)

        last_price = recent['price'].iloc[-1]
        last_usd   = recent['usdprice'].iloc[-1]

        for i, dt in enumerate(future_months, start=1):
            est_price = max(last_price + slope_p * i, 100)
            est_usd   = max(last_usd   + slope_u * i, 0.001)
            estimated_rows.append({
                'date'     : dt,
                'commodity': commodity,
                'price'    : round(est_price, 2),
                'usdprice' : round(est_usd, 6),
                'source'   : 'estimated_trend'
            })
        print(f"    [{commodity}] Linear trend ({n_months} bulan: "
              f"{future_months[0].date()} -> {future_months[-1].date()}) | "
              f"slope = {slope_p:.2f}/bulan")

df_estimated = pd.DataFrame(estimated_rows)
print(f"\n    Total baris estimasi: {len(df_estimated):,}")

# ===========================================================================
# 5. GABUNGKAN SEMUA DATA
# ===========================================================================
print("\n[5] Menggabungkan semua sumber data...")

df_combined = pd.concat([df_nat, df_reconstructed, df_estimated], ignore_index=True)
df_combined['date'] = pd.to_datetime(df_combined['date']).apply(lambda d: d.replace(day=15))
df_combined = df_combined.sort_values(['commodity', 'date']).reset_index(drop=True)

# Deduplikasi: prioritas WFP > Rekonstruksi > Estimasi
df_combined['source_priority'] = df_combined['source'].map(
    lambda s: 0 if s == 'WFP_National_Average'
              else (1 if 'reconstructed' in s else 2)
)
df_combined = (
    df_combined
    .sort_values('source_priority')
    .drop_duplicates(subset=['date', 'commodity'], keep='first')
    .drop(columns='source_priority')
    .sort_values(['commodity', 'date'])
    .reset_index(drop=True)
)

# ===========================================================================
# 6. PIVOT KE FORMAT WIDE (sama seperti Phase 1 output)
# ===========================================================================
print("\n[6] Pivot ke format wide...")

df_wide = df_combined.pivot_table(
    index='date', columns='commodity', values='price', aggfunc='first'
).reset_index()

# Urutkan kolom sesuai TARGET_COMMODITIES
col_order = ['date'] + [c for c in TARGET_COMMODITIES if c in df_wide.columns]
df_wide = df_wide[col_order]
df_wide = df_wide.sort_values('date').reset_index(drop=True)

print(f"    Shape: {df_wide.shape}")
print(f"    Kolom: {list(df_wide.columns)}")

# ===========================================================================
# 7. VALIDASI
# ===========================================================================
print("\n[7] Validasi hasil akhir...")

# Ringkasan sumber per komoditas
print("\n    === Ringkasan sumber data per komoditas ===")
summary_src = df_combined.groupby('commodity').agg(
    first=('date', 'min'),
    last=('date', 'max'),
    rows=('price', 'count'),
    wfp=('source', lambda x: (x == 'WFP_National_Average').sum()),
    reconstructed=('source', lambda x: x.str.contains('reconstructed').sum()),
    estimated=('source', lambda x: x.str.contains('estimated').sum()),
)
summary_src['first'] = summary_src['first'].dt.strftime('%Y-%m')
summary_src['last']  = summary_src['last'].dt.strftime('%Y-%m')
print(summary_src[TARGET_COMMODITIES].to_string() if False else summary_src.to_string())

# Cek nilai beras vs versi lama (skip otomatis pada run pertama)
print("\n    === Perbandingan Harga Beras (Data Baru vs Lama) ===")
_old_path = os.path.join(OUTPUT_DIR, 'national_avg_clean_fixed.csv')
if os.path.exists(_old_path):
    df_old = pd.read_csv(_old_path, parse_dates=['date'])
    rice_old = df_old[df_old['date'] >= '2024-06-01'][['date', 'Rice']].rename(columns={'Rice': 'Rice_lama'})
    rice_new = df_wide[df_wide['date'] >= '2024-06-01'][['date', 'Rice']].rename(columns={'Rice': 'Rice_baru'})
    comparison = pd.merge(rice_old, rice_new, on='date', how='outer').sort_values('date')
    comparison['selisih'] = comparison['Rice_baru'] - comparison['Rice_lama']
    print(comparison.to_string(index=False))
else:
    print("    [SKIP] Belum ada file output sebelumnya (run pertama) -> tidak ada pembanding.")

_removed = {'Fuel (kerosene)', 'Milk (condensed)', 'Wheat flour'}
_found_removed = sorted(_removed.intersection(df_wide.columns))
assert not _found_removed, f"Komoditas yang seharusnya dihapus masih ada: {_found_removed}"
print("    [OK] Output Phase 0 hanya berisi 8 komoditas target pangan.")

# Cek pola kenaikan konstan (bukti tidak ada ekstrapolasi)
print("\n    === Cek Variasi Kenaikan Harga Beras (harus bervariasi, bukan konstan) ===")
rice_vals = df_wide.dropna(subset=['Rice'])['Rice'].values
diffs = np.diff(rice_vals[-24:])
print(f"    Selisih 24 bulan terakhir (std dev): {np.std(diffs):.2f}")
print(f"    -> Jika std dev >> 0 berarti TIDAK ada pola ekstrapolasi konstan")
print(f"    Min diff: {min(diffs):+.2f} | Max diff: {max(diffs):+.2f}")

# ===========================================================================
# 8. SIMPAN OUTPUT
# ===========================================================================
print("\n[8] Menyimpan output...")
df_wide.to_csv(OUTPUT_FILE, index=False)
print(f"    SELESAI! Output: {OUTPUT_FILE}")
print(f"    Baris : {len(df_wide):,}")
print(f"    Kolom : {len(df_wide.columns)}")
print(f"    Periode: {df_wide['date'].min().date()} -> {df_wide['date'].max().date()}")
print()
print("=" * 70)
print(" PHASE 0 (FIXED) SELESAI")
print("=" * 70)


## Tahap: 01_preprocessing.py

In [ ]:
# -*- coding: utf-8 -*-
"""
PHASE 1: PREPROCESSING
=======================
Input  : outputs/national_avg_clean_fixed.csv   (output Phase 0 - sudah wide format)
Output : outputs/national_avg_clean_fixed.csv    (overwrite dengan data bersih)
         outputs/national_avg_enriched.csv       (alias untuk kompatibilitas)

Pipeline:
  1. Load data dari Phase 0 (sudah wide: 1 kolom per komoditas)
  2. Koreksi anomali Jan-2020 (3 komoditas)
  3. Deteksi & hapus outlier (IQR method 5.0x)
  4. Handle missing values: interpolasi linear + ffill/bfill
  5. Simpan ke CSV
"""

import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# PATH SETUP PORTABLE
# ---------------------------------------------------------------------------
def _resolve_project_dir():
    """Ambil root proyek; kalau dijalankan dari folder Code, naik ke folder induk."""
    env_dir = os.environ.get("PROJECT_DIR")
    if env_dir:
        return os.path.abspath(env_dir)

    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()

    base_dir = os.path.abspath(base_dir)
    parent_dir = os.path.abspath(os.path.join(base_dir, os.pardir))

    # Struktur proyek: <PROJECT_DIR>/Code/code.ipynb, <PROJECT_DIR>/dataset, <PROJECT_DIR>/outputs
    # Jadi saat notebook/script dijalankan dari folder Code, output tetap ke folder outputs di root proyek.
    if os.path.basename(base_dir).lower() == "code":
        return parent_dir

    if os.path.isdir(os.path.join(base_dir, "dataset")):
        return base_dir
    if os.path.isdir(os.path.join(parent_dir, "dataset")):
        return parent_dir

    return base_dir

PROJECT_DIR = _resolve_project_dir()
OUTPUT_DIR  = os.path.join(PROJECT_DIR, 'outputs')
FIG_DIR     = os.path.join(OUTPUT_DIR, 'figures')
TAB_DIR     = os.path.join(OUTPUT_DIR, 'tables')

INPUT_FILE    = os.path.join(OUTPUT_DIR, 'national_avg_clean_fixed.csv')
OUTPUT_FILE   = os.path.join(OUTPUT_DIR, 'national_avg_clean_fixed.csv')
ENRICHED_FILE = os.path.join(OUTPUT_DIR, 'national_avg_enriched.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

TARGET_COMMODITIES = [
    "Chili (bird's eye)", "Chili (red)", "Eggs",
    "Meat (beef)", "Meat (chicken, broiler)",
    "Oil (vegetable)", "Rice", "Sugar"
]

print("=" * 70)
print("PHASE 1: PREPROCESSING")
print("=" * 70)

# ===========================================================================
# 1. LOAD DATA (sudah wide format dari Phase 0)
# ===========================================================================
print("\n[1] Loading dataset...")

df_raw = pd.read_csv(INPUT_FILE, parse_dates=['date'])

# Pastikan kolom date ada dan dijadikan index
if 'date' in df_raw.columns:
    df_raw['date'] = pd.to_datetime(df_raw['date'])
    df_raw = df_raw.set_index('date')
else:
    df_raw.index = pd.to_datetime(df_raw.index)

df_raw = df_raw.sort_index()

# Filter hanya komoditas target yang tersedia
available_cols = [c for c in TARGET_COMMODITIES if c in df_raw.columns]
df_pivot = df_raw[available_cols].copy()

print(f"    Total baris      : {len(df_pivot):,}")
print(f"    Rentang tanggal  : {df_pivot.index.min().date()} -> {df_pivot.index.max().date()}")
print(f"    Komoditas        : {len(df_pivot.columns)}")
print(f"    Kolom            : {list(df_pivot.columns)}")

# ===========================================================================
# 2. KOREKSI ANOMALI NATIONAL AVERAGE JANUARI 2020
# ===========================================================================
# Ditemukan 3 anomali pada data WFP National Average:
#
# Commodity                  Des-2019   Jan-2020   Feb-2020
# ----------------------------------------------------------
# Chili (bird's eye)          51,892      6,561     67,821
# Chili (red)                 46,547      6,349     71,312
# Meat (chicken, broiler)     43,860      4,299     43,704
#
# Nilai Januari 2020 sekitar 10x lebih rendah -> diduga kesalahan skala.
# Koreksi: harga dikalikan 10

ANOMALY_COMMODITIES = [
    "Chili (bird's eye)",
    "Chili (red)",
    "Meat (chicken, broiler)"
]

# Coba beberapa kemungkinan tanggal (tanggal-15 atau tanggal-1)
anomaly_date_candidates = [
    pd.Timestamp("2020-01-15"),
    pd.Timestamp("2020-01-01"),
]

n_anomaly = 0
for anom_date in anomaly_date_candidates:
    if anom_date in df_pivot.index:
        for col in ANOMALY_COMMODITIES:
            if col in df_pivot.columns:
                current_val = df_pivot.loc[anom_date, col]
                if not pd.isna(current_val):
                    df_pivot.loc[anom_date, col] *= 10
                    n_anomaly += 1
        break

print(f"    Anomali Jan-2020 dikoreksi: {n_anomaly} sel")

# ===========================================================================
# 3. INTERPOLASI AWAL (isi missing value asli)
# ===========================================================================
df_pivot = df_pivot.interpolate(method='linear', axis=0)

# ===========================================================================
# 4. OUTLIER HANDLING - IQR x5.0 (conservative)
# ===========================================================================
df_pivot_clean = df_pivot.copy()
total_outliers = 0

for col in df_pivot.columns:
    Q1 = df_pivot[col].quantile(0.25)
    Q3 = df_pivot[col].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 5.0 * IQR
    lower_bound = Q1 - 5.0 * IQR

    outlier_mask = (df_pivot[col] < lower_bound) | (df_pivot[col] > upper_bound)
    n_outliers = outlier_mask.sum()

    if n_outliers > 0:
        total_outliers += n_outliers
        df_pivot_clean.loc[outlier_mask, col] = np.nan
        df_pivot_clean[col] = df_pivot_clean[col].interpolate(method='linear')

print(f"\n[INFO] Deteksi outlier - IQR x5.0 (conservative):")
print(f"       Total outlier ditangani: {total_outliers}")

# Sisa missing value di-fill
df_pivot_clean = df_pivot_clean.ffill().bfill()

print(f"\n[OK] Preprocessing selesai. Shape: {df_pivot_clean.shape}")
print(f"     Missing values tersisa: {df_pivot_clean.isnull().sum().sum()}")

# ===========================================================================
# 5. SIMPAN
# ===========================================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_pivot_clean.to_csv(OUTPUT_FILE)          # dengan index (date)
df_pivot_clean.to_csv(ENRICHED_FILE)        # alias untuk kompatibilitas phase berikutnya

print(f"\n[DONE] File tersimpan:")
print(f"       - {OUTPUT_FILE}")
print(f"       - {ENRICHED_FILE}")
print(f"       Shape: {df_pivot_clean.shape}")
print("=" * 70)


## Tahap: 02_eda.py

In [ ]:
"""
PHASE 2: EXPLORATORY DATA ANALYSIS (EDA)
Visualisasi tren, pola musiman, korelasi, volatilitas
"""

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# SETUP PATH PORTABLE
# ============================================================================
def _resolve_project_dir():
    """Ambil root proyek; kalau dijalankan dari folder Code, naik ke folder induk."""
    env_dir = os.environ.get("PROJECT_DIR")
    if env_dir:
        return os.path.abspath(env_dir)

    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()

    base_dir = os.path.abspath(base_dir)
    parent_dir = os.path.abspath(os.path.join(base_dir, os.pardir))

    # Struktur proyek: <PROJECT_DIR>/Code/code.ipynb, <PROJECT_DIR>/dataset, <PROJECT_DIR>/outputs
    # Jadi saat notebook/script dijalankan dari folder Code, output tetap ke folder outputs di root proyek.
    if os.path.basename(base_dir).lower() == "code":
        return parent_dir

    if os.path.isdir(os.path.join(base_dir, "dataset")):
        return base_dir
    if os.path.isdir(os.path.join(parent_dir, "dataset")):
        return parent_dir

    return base_dir

PROJECT_DIR = _resolve_project_dir()
OUTPUT_DIR  = os.path.join(PROJECT_DIR, "outputs")
FIG_DIR     = os.path.join(OUTPUT_DIR, "figures")
TAB_DIR     = os.path.join(OUTPUT_DIR, "tables")
OUTPUT_DATA = OUTPUT_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 10

print("=" * 80)
print("PHASE 2: EXPLORATORY DATA ANALYSIS (EDA)")
print("=" * 80)

# ============================================================================
# 1. LOAD DATA DARI PHASE 1
# ============================================================================
input_file = os.path.join(PROJECT_DIR, "outputs", "national_avg_clean_fixed.csv")
df = pd.read_csv(input_file, parse_dates=['date'], index_col='date')

print(f"\n[OK] Data dimuat: {df.shape}")
print(f"  Periode: {df.index.min().strftime('%B %Y')} - {df.index.max().strftime('%B %Y')}")



# ============================================================================
# 2. LINE PLOT: TREN HARGA SETIAP KOMODITAS (2007-2025)
# ============================================================================
print(f"\nChart [1/6] Membuat: Tren harga historis per komoditas...")

n_items = len(df.columns)
n_cols = 4
n_rows = int(np.ceil(n_items / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.8 * n_cols, 3.8 * n_rows))
axes = np.atleast_1d(axes).flatten()

for idx, col in enumerate(df.columns):
    ax = axes[idx]
    ax.plot(df.index, df[col], linewidth=1.5, color='steelblue', alpha=0.8)
    ax.fill_between(df.index, df[col], alpha=0.2)
    ax.set_title(f'{col}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Harga (IDR)')
    ax.grid(True, alpha=0.3)

    # Format x-axis
    ax.tick_params(axis='x', rotation=45)

# Hapus subplot kosong agar tidak muncul grid kosong
for j in range(len(df.columns), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Tren Harga Komoditas Pangan Nasional (2007-2025)',
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
output_fig1 = os.path.join(PROJECT_DIR, "outputs", "figures", "01_trend_per_komoditas.png")
os.makedirs(os.path.dirname(output_fig1), exist_ok=True)
plt.savefig(output_fig1, dpi=300, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: {output_fig1}")

# ============================================================================
# 3. SEASONAL DECOMPOSITION (untuk 3 komoditas utama)
# ============================================================================
print(f"\nChart [2/6] Membuat: Seasonal decomposition...")
print(f"   Note: Residual besar di akhir adalah karena structural break (inflation)")
print(f"   -> Ini NORMAL dan menunjukkan non-stationarity yang perlu ditangani")

main_commodities = ['Rice', 'Meat (beef)', 'Chili (red)']
fig, axes = plt.subplots(len(main_commodities), 4, figsize=(18, 12))

for row, commodity in enumerate(main_commodities):
    # Interpolate missing values for decomposition
    series_clean = df[commodity].interpolate(method='linear')

    # Decompose
    decomposition = seasonal_decompose(series_clean, model='additive', period=12)

    # Plot: Original
    ax = axes[row, 0]
    decomposition.observed.plot(ax=ax, color='steelblue')
    ax.set_ylabel('Harga (IDR)')
    ax.set_title(f'{commodity} - Original' if row == 0 else '')
    ax.grid(True, alpha=0.3)

    # Plot: Trend
    ax = axes[row, 1]
    decomposition.trend.plot(ax=ax, color='orange')
    ax.set_ylabel('Harga (IDR)')
    ax.set_title(f'{commodity} - Trend' if row == 0 else '')
    ax.grid(True, alpha=0.3)

    # Plot: Seasonal
    ax = axes[row, 2]
    decomposition.seasonal.plot(ax=ax, color='green')
    ax.set_ylabel('Harga (IDR)')
    ax.set_title(f'{commodity} - Seasonal' if row == 0 else '')
    ax.grid(True, alpha=0.3)

    # Plot: Residual
    ax = axes[row, 3]
    decomposition.resid.plot(ax=ax, color='red')
    ax.set_ylabel('Harga (IDR)')
    ax.set_title(f'{commodity} - Residual' if row == 0 else '')
    ax.grid(True, alpha=0.3)

plt.suptitle('Seasonal Decomposition (Additive Model)',
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
output_fig2 = os.path.join(PROJECT_DIR, "outputs", "figures", "02_seasonal_decomposition.png")
plt.savefig(output_fig2, dpi=300, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: {output_fig2}")

# ============================================================================
# 4. HEATMAP KORELASI ANTAR KOMODITAS
# ============================================================================
print(f"\nChart [3/6] Membuat: Heatmap korelasi...")

corr_matrix = df.corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Korelasi Harga Antar Komoditas', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
output_fig3 = os.path.join(PROJECT_DIR, "outputs", "figures", "03_correlation_heatmap.png")
plt.savefig(output_fig3, dpi=300, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: {output_fig3}")

# ============================================================================
# 5. POLA MUSIMAN: BOX PLOT PER BULAN
# ============================================================================
print(f"\nChart [4/6] Membuat: Box plot pola musiman...")

# Tambahkan kolom bulan
df_with_month = df.copy()
df_with_month['month'] = df_with_month.index.month

n_items = len(df.columns)
n_cols = 4
n_rows = int(np.ceil(n_items / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.8 * n_cols, 3.8 * n_rows))
axes = np.atleast_1d(axes).flatten()

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

for idx, col in enumerate(df.columns):
    ax = axes[idx]
    data_by_month = [df_with_month[df_with_month['month'] == m][col].values for m in range(1, 13)]
    bp = ax.boxplot(data_by_month, labels=month_names, patch_artist=True)

    # Color boxes
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')

    ax.set_title(f'{col}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Harga (IDR)')
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=45)

for j in range(len(df.columns), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Pola Musiman: Harga per Bulan (2007-2025)',
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
output_fig4 = os.path.join(PROJECT_DIR, "outputs", "figures", "04_monthly_pattern_boxplot.png")
plt.savefig(output_fig4, dpi=300, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: {output_fig4}")

# ============================================================================
# 6. YEAR-OVER-YEAR GROWTH RATE
# ============================================================================
print(f"\nChart [5/6] Membuat: Year-over-year growth rate...")

df_yoy = df.pct_change(periods=12) * 100  # 12 bulan = 1 tahun

n_items = len(df.columns)
n_cols = 4
n_rows = int(np.ceil(n_items / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.8 * n_cols, 3.8 * n_rows))
axes = np.atleast_1d(axes).flatten()

for idx, col in enumerate(df.columns):
    ax = axes[idx]
    ax.bar(df_yoy.index, df_yoy[col], color='steelblue', alpha=0.7, width=10)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1)
    ax.set_title(f'{col}', fontsize=11, fontweight='bold')
    ax.set_ylabel('YoY Growth (%)')
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=45)

for j in range(len(df.columns), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Year-over-Year (YoY) Growth Rate',
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
output_fig5 = os.path.join(PROJECT_DIR, "outputs", "figures", "05_yoy_growth_rate.png")
plt.savefig(output_fig5, dpi=300, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: {output_fig5}")

# ============================================================================
# 7. ROLLING STATISTICS: VOLATILITAS & MOVING AVERAGE
# ============================================================================
print(f"\nChart [6/6] Membuat: Rolling statistics (volatilitas)...")

n_items = len(df.columns)
n_cols = 4
n_rows = int(np.ceil(n_items / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.8 * n_cols, 3.8 * n_rows))
axes = np.atleast_1d(axes).flatten()

for idx, col in enumerate(df.columns):
    ax = axes[idx]

    # Rolling mean & std
    rolling_mean = df[col].rolling(window=12).mean()
    rolling_std = df[col].rolling(window=12).std()

    # Plot harga
    ax.plot(df.index, df[col], label='Actual Price', color='steelblue', linewidth=1, alpha=0.7)

    # Plot rolling mean
    ax.plot(rolling_mean.index, rolling_mean, label='12-Month MA',
            color='orange', linewidth=2, alpha=0.9)

    # Plot confidence band (mean [00B1] std)
    ax.fill_between(rolling_mean.index,
                     rolling_mean - rolling_std,
                     rolling_mean + rolling_std,
                     alpha=0.2, color='green', label='[00B1]1 Std Dev')

    ax.set_title(f'{col}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Harga (IDR)')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

for j in range(len(df.columns), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Rolling Statistics: 12-Month Moving Average & Volatility',
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
output_fig6 = os.path.join(PROJECT_DIR, "outputs", "figures", "06_rolling_statistics.png")
plt.savefig(output_fig6, dpi=300, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: {output_fig6}")

# ============================================================================
# 8. RINGKASAN STATISTIK
# ============================================================================
print(f"\nChart RINGKASAN STATISTIK (IDR):")
summary_stats = df.describe().T[['count', 'mean', 'std', 'min', 'max']]
print(summary_stats.round(2))

# Simpan ke tabel CSV
output_table = os.path.join(PROJECT_DIR, "outputs", "tables", "eda_summary_statistics.csv")
os.makedirs(os.path.dirname(output_table), exist_ok=True)
summary_stats.to_csv(output_table)
print(f"\n[OK] Tabel tersimpan: {output_table}")

# ============================================================================
# 9. KORELASI TERBAIK & TERBURUK
# ============================================================================
print(f"\nChart TOP 5 KORELASI TERTINGGI:")
# Get upper triangle of correlation matrix
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))

corr_pairs_sorted = sorted(corr_pairs, key=lambda x: abs(x[2]), reverse=True)
for i, (com1, com2, corr) in enumerate(corr_pairs_sorted[:5], 1):
    print(f"  {i}. {com1:25s} [2194] {com2:25s}: {corr:+.3f}")

print(f"\n[OK] PHASE 2 COMPLETE")
print(f"   6 visualisasi tersimpan di: ../outputs/figures/")
print(f"   Statistik ringkasan di: ../outputs/tables/")


## Tahap: 03_stationarity_test.py

In [ ]:
"""
PHASE 3: STATIONARITY TEST
Augmented Dickey-Fuller (ADF) & KPSS test untuk setiap komoditas
Transformasi jika non-stasioner: differencing atau log-transform
"""

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# SETUP PATH PORTABLE
# ============================================================================
def _resolve_project_dir():
    """Ambil root proyek; kalau dijalankan dari folder Code, naik ke folder induk."""
    env_dir = os.environ.get("PROJECT_DIR")
    if env_dir:
        return os.path.abspath(env_dir)

    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()

    base_dir = os.path.abspath(base_dir)
    parent_dir = os.path.abspath(os.path.join(base_dir, os.pardir))

    # Struktur proyek: <PROJECT_DIR>/Code/code.ipynb, <PROJECT_DIR>/dataset, <PROJECT_DIR>/outputs
    # Jadi saat notebook/script dijalankan dari folder Code, output tetap ke folder outputs di root proyek.
    if os.path.basename(base_dir).lower() == "code":
        return parent_dir

    if os.path.isdir(os.path.join(base_dir, "dataset")):
        return base_dir
    if os.path.isdir(os.path.join(parent_dir, "dataset")):
        return parent_dir

    return base_dir

PROJECT_DIR = _resolve_project_dir()
OUTPUT_DIR  = os.path.join(PROJECT_DIR, "outputs")
FIG_DIR     = os.path.join(OUTPUT_DIR, "figures")
TAB_DIR     = os.path.join(OUTPUT_DIR, "tables")
OUTPUT_DATA = OUTPUT_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 10)

print("=" * 80)
print("PHASE 3: STATIONARITY TEST (ADF & KPSS)")
print("=" * 80)

# ============================================================================
# 1. LOAD DATA DARI PHASE 1
# ============================================================================
input_file = os.path.join(PROJECT_DIR, "outputs", "national_avg_clean_fixed.csv")
df = pd.read_csv(input_file, parse_dates=['date'], index_col='date')

print(f"\n[OK] Data dimuat: {df.shape}")

# ============================================================================
# 2. FUNGSI UNTUK STATIONARITY TEST
# ============================================================================
def perform_adf_test(timeseries, name):
    """
    Perform Augmented Dickey-Fuller test
    H0: Series has unit root (non-stationary)
    H1: Series is stationary
    """
    result = adfuller(timeseries.dropna(), autolag='AIC')
    return {
        'commodity': name,
        'adf_statistic': result[0],
        'p_value': result[1],
        'n_lags': result[2],
        'n_obs': result[3],
    }

def perform_kpss_test(timeseries, name):
    """
    Perform KPSS test
    H0: Series is stationary
    H1: Series has unit root (non-stationary)
    """
    result = kpss(timeseries.dropna(), regression='c', nlags='auto')
    return {
        'commodity': name,
        'kpss_statistic': result[0],
        'p_value': result[1],
        'n_lags': result[2],
    }

# ============================================================================
# 3. JALANKAN STATIONARITY TEST
# ============================================================================
print(f"\nChart Menjalankan ADF & KPSS test pada {len(df.columns)} komoditas...")

results_adf = []
results_kpss = []
results_summary = []

for col in df.columns:
    # ADF Test
    adf_result = perform_adf_test(df[col], col)
    results_adf.append(adf_result)

    # KPSS Test
    kpss_result = perform_kpss_test(df[col], col)
    results_kpss.append(kpss_result)

    # Interpretasi
    adf_stationary = adf_result['p_value'] < 0.05
    kpss_stationary = kpss_result['p_value'] >= 0.05  # KPSS: H0 is stationary

    # Consensus
    is_stationary = adf_stationary and kpss_stationary

    results_summary.append({
        'Commodity': col,
        'ADF p-value': adf_result['p_value'],
        'ADF Stationary': 'Yes' if adf_stationary else 'No',
        'KPSS p-value': kpss_result['p_value'],
        'KPSS Stationary': 'Yes' if kpss_stationary else 'No',
        'Is Stationary': 'Yes' if is_stationary else 'No',
        'Transformation': 'None' if is_stationary else 'Differencing',
    })

# ============================================================================
# 4. CREATE SUMMARY TABLE
# ============================================================================
df_summary = pd.DataFrame(results_summary)

print(f"\nChart HASIL STATIONARITY TEST:")
print(df_summary.to_string(index=False))

# ============================================================================
# 5. VISUALISASI: ORIGINAL VS DIFFERENCED
# ============================================================================
print(f"\nChart Membuat visualisasi: Original vs Differenced...")

fig, axes = plt.subplots(len(df.columns), 2, figsize=(16, 16))

for idx, col in enumerate(df.columns):
    is_stationary = df_summary[df_summary['Commodity'] == col]['Is Stationary'].values[0] == 'Yes'

    # Plot original
    ax1 = axes[idx, 0]
    ax1.plot(df.index, df[col], color='steelblue', linewidth=1.5)
    ax1.set_title(f'{col} - Original {"(Stationary)" if is_stationary else "(Non-Stationary)"}',
                  fontsize=11, fontweight='bold')
    ax1.set_ylabel('Harga (IDR)')
    ax1.grid(True, alpha=0.3)

    # Plot differenced (first difference)
    df_diff = df[col].diff().dropna()
    ax2 = axes[idx, 1]
    ax2.plot(df_diff.index, df_diff, color='coral', linewidth=1.5)
    ax2.axhline(y=0, color='red', linestyle='--', linewidth=1)
    ax2.set_title(f'{col} - First Difference', fontsize=11, fontweight='bold')
    ax2.set_ylabel('[0394] Harga (IDR)')
    ax2.grid(True, alpha=0.3)

plt.suptitle('Original vs First Difference', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
output_fig = os.path.join(PROJECT_DIR, "outputs", "figures", "07_stationarity_original_vs_differenced.png")
os.makedirs(os.path.dirname(output_fig), exist_ok=True)
plt.savefig(output_fig, dpi=300, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: {output_fig}")

# ============================================================================
# 6. ANALISIS: BERAPA BANYAK YANG STATIONARY?
# ============================================================================
n_stationary = (df_summary['Is Stationary'] == 'Yes').sum()
n_non_stationary = (df_summary['Is Stationary'] == 'No').sum()

print(f"\nChart ANALISIS STASIONERITAS:")
print(f"  Stationary: {n_stationary}/{len(df_summary)} komoditas")
print(f"  Non-Stationary: {n_non_stationary}/{len(df_summary)} komoditas")

if n_non_stationary > 0:
    print(f"\n  Komoditas yang perlu differencing:")
    non_stat = df_summary[df_summary['Is Stationary'] == 'No']
    for _, row in non_stat.iterrows():
        print(f"    - {row['Commodity']}")

# ============================================================================
# 7. SIMPAN HASIL
# ============================================================================
# Simpan hasil ADF
df_adf = pd.DataFrame(results_adf)
output_adf = os.path.join(PROJECT_DIR, "outputs", "tables", "stationarity_adf_test.csv")
os.makedirs(os.path.dirname(output_adf), exist_ok=True)
df_adf.to_csv(output_adf, index=False)
print(f"\n[OK] ADF test results: {output_adf}")

# Simpan hasil KPSS
df_kpss = pd.DataFrame(results_kpss)
output_kpss = os.path.join(PROJECT_DIR, "outputs", "tables", "stationarity_kpss_test.csv")
df_kpss.to_csv(output_kpss, index=False)
print(f"[OK] KPSS test results: {output_kpss}")

# Simpan ringkasan
output_summary = os.path.join(PROJECT_DIR, "outputs", "tables", "stationarity_summary.csv")
df_summary.to_csv(output_summary, index=False)
print(f"[OK] Summary: {output_summary}")

# ============================================================================
# 8. CREATE DIFFERENCED DATASET (untuk modeling)
# ============================================================================
print(f"\nChart Membuat dataset differenced untuk non-stationary komoditas...")

df_differenced = df.copy()

for col in df_summary[df_summary['Is Stationary'] == 'No']['Commodity'].values:
    df_differenced[col] = df[col].diff()

# Hilangkan NaN dari differencing
df_differenced = df_differenced.dropna()

output_differenced = os.path.join(PROJECT_DIR, "outputs", "national_avg_differenced.csv")
df_differenced.to_csv(output_differenced)
print(f"[OK] Differenced data: {output_differenced}")
print(f"  Shape: {df_differenced.shape}")

# ============================================================================
# 9. VISUALISASI PIE CHART STATIONARITY
# ============================================================================
fig, ax = plt.subplots(figsize=(8, 6))
stationary_counts = [n_stationary, n_non_stationary]
labels = [f'Stationary\n({n_stationary})', f'Non-Stationary\n({n_non_stationary})']
colors = ['#2ecc71', '#e74c3c']
ax.pie(stationary_counts, labels=labels, colors=colors, autopct='%1.1f%%',
       startangle=90, textprops={'fontsize': 12, 'weight': 'bold'})
ax.set_title('Komposisi Stasioneritas Komoditas Pangan',
             fontsize=13, fontweight='bold', pad=20)
output_pie = os.path.join(PROJECT_DIR, "outputs", "figures", "08_stationarity_pie_chart.png")
plt.savefig(output_pie, dpi=300, bbox_inches='tight')
plt.close()
print(f"\n[OK] Pie chart tersimpan: {output_pie}")

# ============================================================================
# 10. DETAIL REPORT
# ============================================================================
print(f"\n{'='*80}")
print(f"DETAIL STATIONARITY REPORT")
print(f"{'='*80}")

for idx, row in df_summary.iterrows():
    print(f"\n{idx+1}. {row['Commodity']}")
    print(f"   ADF p-value: {row['ADF p-value']:.6f} -> {row['ADF Stationary']}")
    print(f"   KPSS p-value: {row['KPSS p-value']:.6f} -> {row['KPSS Stationary']}")
    print(f"   Consensus: {row['Is Stationary']}")
    print(f"   Recommendation: {row['Transformation']}")

# ============================================================================
# 11. ACF/PACF PLOT (Optional: untuk 3 komoditas utama)
# ============================================================================
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

print(f"\nChart Membuat ACF/PACF plot untuk 3 komoditas utama...")

main_commodities = ['Rice', 'Meat (beef)', 'Chili (red)']
fig, axes = plt.subplots(3, 2, figsize=(14, 10))

for row, commodity in enumerate(main_commodities):
    # ACF
    plot_acf(df[commodity].dropna(), lags=40, ax=axes[row, 0], title=f'{commodity} - ACF')

    # PACF
    plot_pacf(df[commodity].dropna(), lags=40, ax=axes[row, 1], title=f'{commodity} - PACF')

plt.suptitle('Autocorrelation (ACF) & Partial Autocorrelation (PACF)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
output_acf = os.path.join(PROJECT_DIR, "outputs", "figures", "09_acf_pacf_plots.png")
plt.savefig(output_acf, dpi=300, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: {output_acf}")

print(f"\n[OK] PHASE 3 COMPLETE")
print(f"   Stationarity test results tersimpan di: ../outputs/tables/")
print(f"   Visualisasi tersimpan di: ../outputs/figures/")
print(f"   Dataset clean & differenced siap untuk modeling!")


## Tahap: 04_sarima_model.py

In [ ]:
"""
PHASE 4: SARIMA MODELING
Seasonal ARIMA per komoditas  -  auto parameter tuning, fitting, diagnostics, dan forecast 2026-2030

Strategi:
- Semua 11 komoditas non-stationary -> gunakan d=1 (first differencing)
- LOG-TRANSFORM diterapkan sebelum fitting:
    * Model difit pada log(harga), forecast di-back-transform dengan exp()
    * Menjamin semua forecast > 0 (tidak ada harga negatif)
    * Lebih tepat untuk data harga dengan variansi yang tumbuh (heteroscedastic)
- Auto-search parameter (p,d,q)(P,D,Q,s=12) via AIC minimization
- Fit pada data training (Jan 2007 - Des 2021)
- Evaluasi pada test set (Jan 2022 - Mei 2025) dalam skala IDR asli
- Forecast short-term: Jun 2025 - Des 2030 (67 bulan)
"""

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import itertools
from datetime import datetime

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

# ============================================================================
# SETUP PATH PORTABLE
# ============================================================================
def _resolve_project_dir():
    """Ambil root proyek; kalau dijalankan dari folder Code, naik ke folder induk."""
    env_dir = os.environ.get("PROJECT_DIR")
    if env_dir:
        return os.path.abspath(env_dir)

    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()

    base_dir = os.path.abspath(base_dir)
    parent_dir = os.path.abspath(os.path.join(base_dir, os.pardir))

    # Struktur proyek: <PROJECT_DIR>/Code/code.ipynb, <PROJECT_DIR>/dataset, <PROJECT_DIR>/outputs
    # Jadi saat notebook/script dijalankan dari folder Code, output tetap ke folder outputs di root proyek.
    if os.path.basename(base_dir).lower() == "code":
        return parent_dir

    if os.path.isdir(os.path.join(base_dir, "dataset")):
        return base_dir
    if os.path.isdir(os.path.join(parent_dir, "dataset")):
        return parent_dir

    return base_dir

PROJECT_DIR = _resolve_project_dir()
OUTPUT_DATA    = os.path.join(PROJECT_DIR, "outputs")
OUTPUT_FIGURES = os.path.join(OUTPUT_DATA, "figures")
OUTPUT_TABLES  = os.path.join(OUTPUT_DATA, "tables")

os.makedirs(OUTPUT_DATA, exist_ok=True)
os.makedirs(OUTPUT_FIGURES, exist_ok=True)
os.makedirs(OUTPUT_TABLES,  exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150

print("=" * 80)
print("PHASE 4: SARIMA MODELING")
print("=" * 80)

# ============================================================================
# 1. LOAD DATA CLEAN
# ============================================================================
input_file = os.path.join(OUTPUT_DATA, "national_avg_clean_fixed.csv")
df = pd.read_csv(input_file, parse_dates=['date'], index_col='date')

# Data punya tanggal ke-15 setiap bulan -> normalize ke awal bulan (MS)
# agar SARIMAX bisa mendeteksi frekuensi dengan benar
df.index = pd.to_datetime(df.index).to_period('M').to_timestamp()  # -> tanggal 1 setiap bulan
df = df.sort_index()

# Reindex ke range penuh (ada 1 missing date: Jul 2017) lalu forward-fill
full_idx = pd.date_range(df.index[0], df.index[-1], freq='MS')
df = df.reindex(full_idx).ffill()
df.index.freq = pd.tseries.frequencies.to_offset('MS')

print(f"\n[OK] Data dimuat: {df.shape}")
print(f"     Rentang: {df.index[0].strftime('%b %Y')} - {df.index[-1].strftime('%b %Y')}")
print(f"     Komoditas: {list(df.columns)}\n")

COMMODITIES = list(df.columns)

# ============================================================================
# 2. TRAIN / TEST SPLIT
# ============================================================================
TRAIN_END   = '2021-12-01'
TEST_START  = '2022-01-01'
TEST_END    = df.index[-1]

# Forecast horizon: setelah data terakhir s.d. Des 2030
LAST_DATA_DATE  = df.index[-1]
FORECAST_END    = pd.Timestamp('2030-12-01')
FORECAST_STEPS  = len(pd.date_range(LAST_DATA_DATE, FORECAST_END, freq='MS')) - 1

print(f"[INFO] Train  : Jan 2007 - {TRAIN_END}")
print(f"[INFO] Test   : {TEST_START} - {TEST_END.strftime('%Y-%m-%d')}")
print(f"[INFO] Forecast: {(LAST_DATA_DATE + pd.DateOffset(months=1)).strftime('%b %Y')} - Des 2030")
print(f"         Jumlah langkah forecast: {FORECAST_STEPS} bulan\n")

df_train = df.loc[:TRAIN_END]
df_test  = df.loc[TEST_START:TEST_END]

# ============================================================================
# 3. FUNGSI UTILITAS
# ============================================================================

def mape(y_true, y_pred):
    """Mean Absolute Percentage Error  -  hindari pembagian nol."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def auto_sarima_search(series, d=1, D=1, s=12, verbose=False):
    """
    Grid search SARIMA order berdasarkan AIC minimum.
    Mengembalikan (order, seasonal_order, aic).
    """
    best_aic    = np.inf
    best_order  = (1, d, 1)
    best_sorder = (0, D, 1, s)

    p_list = [0, 1, 2]
    q_list = [0, 1, 2]
    P_list = [0, 1]
    Q_list = [0, 1]

    combos = list(itertools.product(p_list, q_list, P_list, Q_list))

    for (p, q, P, Q) in combos:
        # skip model tanpa parameter AR/MA sama sekali
        if p == 0 and q == 0 and P == 0 and Q == 0:
            continue
        try:
            model = SARIMAX(
                series,
                order=(p, d, q),
                seasonal_order=(P, D, Q, s),
                enforce_stationarity=False,
                enforce_invertibility=False,
                trend='n'
            )
            result = model.fit(disp=False, maxiter=150, method='lbfgs')
            if result.aic < best_aic:
                best_aic    = result.aic
                best_order  = (p, d, q)
                best_sorder = (P, D, Q, s)
        except Exception:
            continue

    return best_order, best_sorder, best_aic


def fit_sarima(series, order, seasonal_order):
    """Fit SARIMA model dan kembalikan result."""
    model = SARIMAX(
        series,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
        trend='n'
    )
    return model.fit(disp=False, maxiter=300, method='lbfgs')


# ============================================================================
# 4. PARAMETER TUNING + FITTING PER KOMODITAS
# ============================================================================
print("=" * 80)
print("AUTO ARIMA PARAMETER SEARCH (grid search via AIC)")
print("=" * 80)

sarima_results   = {}   # {commodity: fitted_result}
sarima_params    = []   # tabel parameter
eval_metrics     = []   # tabel evaluasi
forecast_records = []   # tabel forecast

for commodity in COMMODITIES:
    print(f"\n[>] {commodity}")

    train_series = df_train[commodity].dropna()
    test_series  = df_test[commodity].dropna()

    # --- LOG-TRANSFORM: fit pada log(harga) untuk jamin forecast > 0 ---
    log_train = np.log(train_series)
    log_full  = np.log(df[commodity].dropna())

    # --- Parameter search (pada log-series) ---
    print(f"    Mencari parameter terbaik (grid search)...", end="", flush=True)
    order, sorder, aic = auto_sarima_search(log_train, d=1, D=1, s=12)
    print(f" selesai.")
    aic_str = f'{aic:.2f}' if aic != np.inf else 'N/A'
    print(f"    Best order  : SARIMA{order}x{sorder}  AIC={aic_str} (log-scale)")

    # --- Fit pada training set (log-scale) ---
    try:
        result = fit_sarima(log_train, order, sorder)
    except Exception as e:
        print(f"    [WARN] Fit gagal ({e}), fallback ke SARIMA(1,1,1)(0,1,1,12)")
        order  = (1, 1, 1)
        sorder = (0, 1, 1, 12)
        try:
            result = fit_sarima(log_train, order, sorder)
        except Exception as e2:
            print(f"    [ERROR] Fallback juga gagal: {e2}. Skip komoditas ini.")
            continue

    sarima_results[commodity] = result

    # Simpan parameter
    sarima_params.append({
        'Commodity'      : commodity,
        'p'              : order[0],
        'd'              : order[1],
        'q'              : order[2],
        'P'              : sorder[0],
        'D'              : sorder[1],
        'Q'              : sorder[2],
        'S'              : sorder[3],
        'AIC_log'        : round(aic, 2),
        'BIC_log'        : round(result.bic, 2),
        'transform'      : 'log',
    })

    # --- Prediksi di test set (back-transform ke IDR) ---
    n_test = len(test_series)
    if n_test > 0:
        try:
            log_forecast_test = result.forecast(steps=n_test)
            log_forecast_test.index = test_series.index

            # Back-transform: exp(log_forecast) -> IDR
            forecast_test = np.exp(log_forecast_test)

            mae_val  = mean_absolute_error(test_series, forecast_test)
            rmse_val = np.sqrt(mean_squared_error(test_series, forecast_test))
            mape_val = mape(test_series.values, forecast_test.values)

            print(f"    MAE={mae_val:,.0f}  RMSE={rmse_val:,.0f}  MAPE={mape_val:.2f}%  [IDR scale]")

            eval_metrics.append({
                'Commodity' : commodity,
                'Model'     : 'SARIMA (log)',
                'MAE'       : round(mae_val, 2),
                'RMSE'      : round(rmse_val, 2),
                'MAPE_%'    : round(mape_val, 2),
                'n_test'    : n_test,
            })
        except Exception as e:
            print(f"    [WARN] Evaluasi test set gagal: {e}")
            eval_metrics.append({
                'Commodity' : commodity,
                'Model'     : 'SARIMA (log)',
                'MAE'       : np.nan,
                'RMSE'      : np.nan,
                'MAPE_%'    : np.nan,
                'n_test'    : n_test,
            })
    else:
        print("    [WARN] Test set kosong, skip evaluasi.")

    # --- Re-fit di SELURUH data log (untuk forecast masa depan) ---
    try:
        result_full = fit_sarima(log_full, order, sorder)
    except Exception:
        result_full = result  # fallback ke training-only fit

    # --- Forecast masa depan (back-transform ke IDR) ---
    if FORECAST_STEPS > 0:
        try:
            fc_obj  = result_full.get_forecast(steps=FORECAST_STEPS)
            fc_log  = fc_obj.predicted_mean          # log-scale
            fc_ci_log = fc_obj.conf_int(alpha=0.05)  # log-scale CI

            # Back-transform dengan exp()  -  menjamin semua nilai > 0
            fc_mean = np.exp(fc_log)
            fc_lower = np.exp(fc_ci_log.iloc[:, 0])
            fc_upper = np.exp(fc_ci_log.iloc[:, 1])

            fc_dates = pd.date_range(
                start=LAST_DATA_DATE + pd.DateOffset(months=1),
                periods=FORECAST_STEPS,
                freq='MS'
            )
            fc_mean.index  = fc_dates
            fc_lower.index = fc_dates
            fc_upper.index = fc_dates

            for i, dt in enumerate(fc_dates):
                forecast_records.append({
                    'date'      : dt,
                    'commodity' : commodity,
                    'forecast'  : round(float(fc_mean.iloc[i]), 2),
                    'lower_95'  : round(float(fc_lower.iloc[i]), 2),
                    'upper_95'  : round(float(fc_upper.iloc[i]), 2),
                    'model'     : 'SARIMA',
                })
        except Exception as e:
            print(f"    [WARN] Forecast gagal: {e}")

print("\n[OK] Semua komoditas selesai diproses.\n")

# ============================================================================
# 5. SIMPAN HASIL KE CSV
# ============================================================================

# 5a. Parameter tabel
df_params = pd.DataFrame(sarima_params)
params_file = os.path.join(OUTPUT_TABLES, "sarima_parameters.csv")
df_params.to_csv(params_file, index=False)
print(f"[OK] Parameter SARIMA  : {params_file}")

# 5b. Evaluation tabel
df_eval = pd.DataFrame(eval_metrics)
eval_file = os.path.join(OUTPUT_TABLES, "sarima_evaluation.csv")
df_eval.to_csv(eval_file, index=False)
print(f"[OK] Evaluasi SARIMA   : {eval_file}")

# 5c. Forecast tabel
df_forecast = pd.DataFrame(forecast_records)
forecast_file = os.path.join(OUTPUT_TABLES, "sarima_forecast_2026_2030.csv")
df_forecast.to_csv(forecast_file, index=False)
print(f"[OK] Forecast 2026-2030: {forecast_file}")

# ============================================================================
# 6. VISUALISASI  -  FORECAST vs ACTUAL PER KOMODITAS
# ============================================================================
print("\n[>] Membuat visualisasi forecast vs actual...")

# Buat df_forecast jika kosong agar tidak error
if len(forecast_records) == 0:
    df_forecast = pd.DataFrame(columns=['date','commodity','forecast','lower_95','upper_95','model'])
else:
    df_forecast = pd.DataFrame(forecast_records)
    df_forecast['date'] = pd.to_datetime(df_forecast['date'])

n_cols = 2
n_rows = (len(COMMODITIES) + 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for idx, commodity in enumerate(COMMODITIES):
    ax = axes[idx]

    # Data historis (full)
    hist = df[commodity]

    # Forecast masa depan
    fc_df = df_forecast[df_forecast['commodity'] == commodity].copy()
    if len(fc_df) > 0:
        fc_df['date'] = pd.to_datetime(fc_df['date'])
        fc_df = fc_df.sort_values('date')

    # Plot historis
    ax.plot(hist.index, hist.values, color='#2C3E50', linewidth=1.5,
            label='Historical', alpha=0.9)

    # Plot test-set prediction
    result_train = sarima_results.get(commodity)
    test_series  = df_test[commodity].dropna()
    if result_train is not None and len(test_series) > 0:
        try:
            pred_test_log = result_train.forecast(steps=len(test_series))
            pred_test_log.index = test_series.index
            # Back-transform dari log-scale ke IDR (model difit pada log data)
            pred_test_idr = np.exp(pred_test_log)
            ax.plot(pred_test_idr.index, pred_test_idr.values,
                    color='#F39C12', linewidth=1.5, linestyle='--',
                    label='Pred. (Test)', alpha=0.85)
        except Exception:
            pass

    # Plot forecast
    if len(fc_df) > 0:
        ax.plot(fc_df['date'], fc_df['forecast'],
                color='#E74C3C', linewidth=2, label='Forecast 2026-2030')
        ax.fill_between(
            fc_df['date'],
            fc_df['lower_95'],
            fc_df['upper_95'],
            color='#E74C3C', alpha=0.15, label='CI 95%'
        )

    # Train/Test garis pemisah
    ax.axvline(pd.Timestamp(TRAIN_END), color='gray', linestyle=':', linewidth=1,
               label=f'Train/Test split')

    ax.set_title(f'{commodity}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Harga (IDR)', fontsize=9)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True, alpha=0.3)

# Hapus subplot kosong agar tidak muncul grid kosong
for j in range(len(COMMODITIES), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('SARIMA  -  Forecast vs Actual (2007-2030)', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()

fig_file = os.path.join(OUTPUT_FIGURES, "10_sarima_forecast_vs_actual.png")
plt.savefig(fig_file, dpi=150, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: {fig_file}")

# ============================================================================
# 7. VISUALISASI  -  FORECAST SHORT-TERM (ZOOM 2020-2030)
# ============================================================================
print("[>] Membuat visualisasi forecast short-term (zoom 2020-2030)...")

fig2, axes2 = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes2 = axes2.flatten()

for idx, commodity in enumerate(COMMODITIES):
    ax = axes2[idx]

    # Zoom historis 2020 ke atas
    hist = df[commodity].loc['2020-01-01':]

    fc_df = df_forecast[df_forecast['commodity'] == commodity].copy()
    if len(fc_df) > 0:
        fc_df['date'] = pd.to_datetime(fc_df['date'])
        fc_df = fc_df.sort_values('date')

    ax.plot(hist.index, hist.values, color='#2C3E50', linewidth=2,
            label='Historis', alpha=0.9)

    if len(fc_df) > 0:
        ax.plot(fc_df['date'], fc_df['forecast'],
                color='#E74C3C', linewidth=2.5, label='Forecast SARIMA')
        ax.fill_between(
            fc_df['date'],
            fc_df['lower_95'],
            fc_df['upper_95'],
            color='#E74C3C', alpha=0.2, label='CI 95%'
        )

    ax.axvline(LAST_DATA_DATE, color='#27AE60', linestyle='--', linewidth=1.5,
               label='Batas data')

    ax.set_title(f'{commodity}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Harga (IDR)', fontsize=9)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True, alpha=0.3)

for j in range(len(COMMODITIES), len(axes2)):
    fig2.delaxes(axes2[j])

plt.suptitle('SARIMA  -  Forecast Harga Pangan 2026-2030\n(Zoom 2020-2030)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()

fig2_file = os.path.join(OUTPUT_FIGURES, "11_sarima_forecast_2026_2030.png")
plt.savefig(fig2_file, dpi=150, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: {fig2_file}")

# ============================================================================
# 8. VISUALISASI  -  DIAGNOSTICS PLOT (untuk 3 komoditas utama)
# ============================================================================
print("[>] Membuat diagnostics plot (residual)...")

main_commodities = ['Rice', 'Meat (beef)', 'Chili (red)']
fig3, axes3 = plt.subplots(len(main_commodities), 4, figsize=(20, 4 * len(main_commodities)))

for row_idx, commodity in enumerate(main_commodities):
    result = sarima_results.get(commodity)
    if result is None:
        continue

    residuals = result.resid.dropna()

    # Residual time series
    ax = axes3[row_idx, 0]
    ax.plot(residuals, color='steelblue', linewidth=1)
    ax.axhline(0, color='red', linewidth=1, linestyle='--')
    ax.set_title(f'{commodity}  -  Residuals', fontsize=10, fontweight='bold')
    ax.set_ylabel('Residual (log-scale)')
    ax.grid(True, alpha=0.3)

    # Histogram residual
    ax = axes3[row_idx, 1]
    ax.hist(residuals, bins=25, color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_title(f'{commodity}  -  Distribusi Residual', fontsize=10, fontweight='bold')
    ax.set_xlabel('Residual')
    ax.grid(True, alpha=0.3)

    # ACF residual
    ax = axes3[row_idx, 2]
    plot_acf(residuals, lags=30, ax=ax, title=f'{commodity}  -  ACF Residual')
    ax.grid(True, alpha=0.3)

    # PACF residual
    ax = axes3[row_idx, 3]
    plot_pacf(residuals, lags=30, ax=ax, title=f'{commodity}  -  PACF Residual')
    ax.grid(True, alpha=0.3)

plt.suptitle('SARIMA  -  Diagnostics Residual (Rice, Meat (beef), Chili (red))',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()

fig3_file = os.path.join(OUTPUT_FIGURES, "12_sarima_diagnostics.png")
plt.savefig(fig3_file, dpi=150, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: {fig3_file}")

# ============================================================================
# 9. VISUALISASI  -  EVALUASI METRIC COMPARISON (Bar Chart)
# ============================================================================
print("[>] Membuat bar chart evaluasi MAPE per komoditas...")

df_eval_sorted = df_eval.sort_values('MAPE_%')

fig4, axes4 = plt.subplots(1, 3, figsize=(18, 6))

metrics_info = [
    ('MAE',    'MAE (IDR)', '#3498DB'),
    ('RMSE',   'RMSE (IDR)', '#E74C3C'),
    ('MAPE_%', 'MAPE (%)',  '#2ECC71'),
]

for ax_idx, (metric, label, color) in enumerate(metrics_info):
    df_sorted = df_eval.sort_values(metric)
    bars = axes4[ax_idx].barh(
        df_sorted['Commodity'], df_sorted[metric],
        color=color, alpha=0.85, edgecolor='white'
    )
    axes4[ax_idx].set_xlabel(label, fontsize=11)
    axes4[ax_idx].set_title(f'{label} per Komoditas', fontsize=12, fontweight='bold')
    axes4[ax_idx].grid(True, axis='x', alpha=0.3)

    # Label nilai
    for bar in bars:
        width = bar.get_width()
        fmt = f'{width:,.0f}' if metric != 'MAPE_%' else f'{width:.2f}%'
        axes4[ax_idx].text(
            width * 1.01, bar.get_y() + bar.get_height() / 2,
            fmt, va='center', fontsize=8
        )

plt.suptitle('SARIMA  -  Evaluasi Model (Test Set Jan 2022 - Mei 2025)',
             fontsize=13, fontweight='bold')
plt.tight_layout()

fig4_file = os.path.join(OUTPUT_FIGURES, "13_sarima_evaluation_metrics.png")
plt.savefig(fig4_file, dpi=150, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: {fig4_file}")

# ============================================================================
# 10. PRINT SUMMARY AKHIR
# ============================================================================
print("\n" + "=" * 80)
print("RINGKASAN HASIL SARIMA")
print("=" * 80)

print("\n--- PARAMETER TERBAIK ---")
print(df_params.to_string(index=False))

print("\n--- EVALUASI (Test Set) ---")
print(df_eval.sort_values('MAPE_%').to_string(index=False))

print(f"\n--- STATISTIK FORECAST (Jun 2025 - Des 2030) ---")
fc_wide = df_forecast.pivot(index='date', columns='commodity', values='forecast')
print(f"  Forecast tersimpan: {forecast_file}")
print(f"  Shape             : {fc_wide.shape}")

# Preview harga proyeksi Des 2030
dec2030 = df_forecast[
    df_forecast['date'].astype(str).str.startswith('2030-12')
][['commodity', 'forecast', 'lower_95', 'upper_95']].sort_values('commodity')

if len(dec2030) > 0:
    print("\n  Proyeksi Harga  -  Desember 2030 (IDR):")
    for _, row in dec2030.iterrows():
        print(f"    {row['commodity']:<30} {row['forecast']:>12,.0f}  "
              f"(CI: {row['lower_95']:,.0f} - {row['upper_95']:,.0f})")

print("""
[DONE] PHASE 4 COMPLETE
   Files tersimpan:
     - outputs/tables/sarima_parameters.csv
     - outputs/tables/sarima_evaluation.csv
     - outputs/tables/sarima_forecast_2026_2030.csv
     - outputs/figures/10_sarima_forecast_vs_actual.png
     - outputs/figures/11_sarima_forecast_2026_2030.png
     - outputs/figures/12_sarima_diagnostics.png
     - outputs/figures/13_sarima_evaluation_metrics.png

   Next Step --> Phase 5: ETS (Exponential Smoothing) Modeling
""")


## Tahap: 05_ets_model.py

In [ ]:
"""
PHASE 5: ETS (EXPONENTIAL SMOOTHING) MODELING
Holt-Winters Exponential Smoothing per komoditas

Strategi:
- Model difit langsung pada harga asli (IDR)  -  ETS handles multiplicative
  seasonality secara native, tidak perlu log-transform
- Grid search: trend x seasonal x damped_trend
- Evaluasi pada test set (Jan 2022 - Mei 2025)
- Forecast short-term: Jun 2025 - Des 2030 (67 bulan)
- Hasil dibandingkan dengan SARIMA (Phase 4) di Phase 7
"""

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import itertools

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

# ============================================================================
# SETUP PATH PORTABLE
# ============================================================================
def _resolve_project_dir():
    """Ambil root proyek; kalau dijalankan dari folder Code, naik ke folder induk."""
    env_dir = os.environ.get("PROJECT_DIR")
    if env_dir:
        return os.path.abspath(env_dir)

    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()

    base_dir = os.path.abspath(base_dir)
    parent_dir = os.path.abspath(os.path.join(base_dir, os.pardir))

    # Struktur proyek: <PROJECT_DIR>/Code/code.ipynb, <PROJECT_DIR>/dataset, <PROJECT_DIR>/outputs
    # Jadi saat notebook/script dijalankan dari folder Code, output tetap ke folder outputs di root proyek.
    if os.path.basename(base_dir).lower() == "code":
        return parent_dir

    if os.path.isdir(os.path.join(base_dir, "dataset")):
        return base_dir
    if os.path.isdir(os.path.join(parent_dir, "dataset")):
        return parent_dir

    return base_dir

PROJECT_DIR = _resolve_project_dir()
OUTPUT_DATA    = os.path.join(PROJECT_DIR, "outputs")
OUTPUT_FIGURES = os.path.join(OUTPUT_DATA, "figures")
OUTPUT_TABLES  = os.path.join(OUTPUT_DATA, "tables")

os.makedirs(OUTPUT_DATA, exist_ok=True)
os.makedirs(OUTPUT_FIGURES, exist_ok=True)
os.makedirs(OUTPUT_TABLES,  exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150

print("=" * 80)
print("PHASE 5: ETS (EXPONENTIAL SMOOTHING) MODELING")
print("=" * 80)

# ============================================================================
# 1. LOAD DATA CLEAN
# ============================================================================
input_file = os.path.join(OUTPUT_DATA, "national_avg_clean_fixed.csv")
df = pd.read_csv(input_file, parse_dates=['date'], index_col='date')

df.index = pd.to_datetime(df.index).to_period('M').to_timestamp()
df = df.sort_index()
full_idx = pd.date_range(df.index[0], df.index[-1], freq='MS')
df = df.reindex(full_idx).ffill()
df.index.freq = pd.tseries.frequencies.to_offset('MS')

print(f"\n[OK] Data dimuat: {df.shape}")
print(f"     Rentang: {df.index[0].strftime('%b %Y')} - {df.index[-1].strftime('%b %Y')}")
print(f"     Komoditas: {list(df.columns)}\n")

COMMODITIES = list(df.columns)

# ============================================================================
# 2. TRAIN / TEST SPLIT
# ============================================================================
TRAIN_END      = '2021-12-01'
TEST_START     = '2022-01-01'
TEST_END       = df.index[-1]
LAST_DATA_DATE = df.index[-1]
FORECAST_END   = pd.Timestamp('2030-12-01')
FORECAST_STEPS = len(pd.date_range(LAST_DATA_DATE, FORECAST_END, freq='MS')) - 1

print(f"[INFO] Train   : Jan 2007 - {TRAIN_END}")
print(f"[INFO] Test    : {TEST_START} - {TEST_END.strftime('%Y-%m-%d')}")
print(f"[INFO] Forecast: {(LAST_DATA_DATE + pd.DateOffset(months=1)).strftime('%b %Y')} - Des 2030")
print(f"       Jumlah langkah forecast: {FORECAST_STEPS} bulan\n")

df_train = df.loc[:TRAIN_END]
df_test  = df.loc[TEST_START:TEST_END]

# ============================================================================
# 3. FUNGSI UTILITAS
# ============================================================================

def mape(y_true, y_pred):
    """Mean Absolute Percentage Error  -  aman terhadap nilai 0."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def fit_ets(series, trend, seasonal, damped, seasonal_periods=12):
    """
    Fit Holt-Winters ETS model.
    Mengembalikan fitted result atau None jika gagal.
    """
    try:
        model = ExponentialSmoothing(
            series,
            trend=trend,
            seasonal=seasonal,
            seasonal_periods=seasonal_periods if seasonal else None,
            damped_trend=damped,
            initialization_method='estimated'
        )
        result = model.fit(optimized=True, remove_bias=True)
        return result
    except Exception:
        return None


def auto_ets_search(series, seasonal_periods=12):
    """
    Grid search konfigurasi ETS berdasarkan AIC minimum.
    Kembalikan (config_dict, aic, fitted_result).
    """
    # Kombinasi yang akan dicoba
    trend_opts   = ['add', 'mul', None]
    seasonal_opts = ['add', 'mul', None]
    damped_opts  = [True, False]

    best_aic    = np.inf
    best_config = {'trend': 'add', 'seasonal': 'add', 'damped': False}
    best_result = None

    for trend, seasonal, damped in itertools.product(trend_opts, seasonal_opts, damped_opts):
        # damped hanya relevan jika ada trend
        if damped and trend is None:
            continue
        # seasonal=mul membutuhkan semua nilai > 0
        if seasonal == 'mul' and (series <= 0).any():
            continue
        # trend=mul membutuhkan semua nilai > 0
        if trend == 'mul' and (series <= 0).any():
            continue

        result = fit_ets(series, trend, seasonal, damped, seasonal_periods)
        if result is None:
            continue

        try:
            aic = result.aic
            if aic < best_aic:
                best_aic    = aic
                best_config = {'trend': trend, 'seasonal': seasonal, 'damped': damped}
                best_result = result
        except Exception:
            continue

    return best_config, best_aic, best_result


# ============================================================================
# 4. ETS FITTING PER KOMODITAS
# ============================================================================
print("=" * 80)
print("AUTO ETS SEARCH (grid search via AIC)")
print("=" * 80)

ets_results_train = {}  # {commodity: fitted_result on train}
ets_params        = []
eval_metrics      = []
forecast_records  = []

for commodity in COMMODITIES:
    print(f"\n[>] {commodity}")

    train_series = df_train[commodity].dropna()
    test_series  = df_test[commodity].dropna()

    # --- Grid search pada training set ---
    print(f"    Mencari konfigurasi terbaik...", end="", flush=True)
    config, aic, result = auto_ets_search(train_series)
    print(f" selesai.")

    if result is None:
        print(f"    [ERROR] Semua konfigurasi ETS gagal. Skip.")
        continue

    trend_str    = config['trend']    if config['trend']    else 'None'
    seasonal_str = config['seasonal'] if config['seasonal'] else 'None'
    damped_str   = 'damped' if config['damped'] else 'not_damped'
    aic_str      = f'{aic:.2f}' if aic != np.inf else 'N/A'

    print(f"    Best config : trend={trend_str}, seasonal={seasonal_str}, {damped_str}")
    print(f"    AIC         : {aic_str}")

    ets_results_train[commodity] = result

    # Simpan parameter
    ets_params.append({
        'Commodity'       : commodity,
        'trend'           : trend_str,
        'seasonal'        : seasonal_str,
        'damped'          : config['damped'],
        'seasonal_periods': 12,
        'AIC'             : round(aic, 2),
        'BIC'             : round(result.bic, 2) if hasattr(result, 'bic') else np.nan,
        'SSE'             : round(result.sse, 2),
    })

    # --- Evaluasi pada test set ---
    n_test = len(test_series)
    if n_test > 0:
        try:
            pred_test = result.forecast(n_test)
            pred_test.index = test_series.index

            # Clip agar tidak ada nilai negatif (ETS additive bisa negatif untuk volatile series)
            pred_test = pred_test.clip(lower=0)

            mae_val  = mean_absolute_error(test_series, pred_test)
            rmse_val = np.sqrt(mean_squared_error(test_series, pred_test))
            mape_val = mape(test_series.values, pred_test.values)

            print(f"    MAE={mae_val:,.0f}  RMSE={rmse_val:,.0f}  MAPE={mape_val:.2f}%")

            eval_metrics.append({
                'Commodity': commodity,
                'Model'    : 'ETS',
                'MAE'      : round(mae_val, 2),
                'RMSE'     : round(rmse_val, 2),
                'MAPE_%'   : round(mape_val, 2),
                'n_test'   : n_test,
            })
        except Exception as e:
            print(f"    [WARN] Evaluasi gagal: {e}")
            eval_metrics.append({
                'Commodity': commodity, 'Model': 'ETS',
                'MAE': np.nan, 'RMSE': np.nan, 'MAPE_%': np.nan, 'n_test': n_test,
            })
    else:
        print("    [WARN] Test set kosong, skip evaluasi.")

    # --- Re-fit pada seluruh data (untuk forecast masa depan) ---
    _, _, result_full = auto_ets_search(df[commodity].dropna())
    if result_full is None:
        result_full = result  # fallback

    # --- Forecast masa depan ---
    if FORECAST_STEPS > 0:
        try:
            fc_mean = result_full.forecast(FORECAST_STEPS)
            fc_dates = pd.date_range(
                start=LAST_DATA_DATE + pd.DateOffset(months=1),
                periods=FORECAST_STEPS,
                freq='MS'
            )
            fc_mean.index = fc_dates

            # Simulasi CI: gunakan in-sample residual std sebagai proxy
            resid_std = np.std(result_full.resid.dropna())

            for i, dt in enumerate(fc_dates):
                step = i + 1
                # CI melebar seiring horizon (proporsional sqrt(step))
                margin = 1.96 * resid_std * np.sqrt(step)
                fc_val = max(float(fc_mean.iloc[i]), 0)  # clip negatif

                forecast_records.append({
                    'date'     : dt,
                    'commodity': commodity,
                    'forecast' : round(fc_val, 2),
                    'lower_95' : round(max(fc_val - margin, 0), 2),
                    'upper_95' : round(fc_val + margin, 2),
                    'model'    : 'ETS',
                })
        except Exception as e:
            print(f"    [WARN] Forecast gagal: {e}")

print("\n[OK] Semua komoditas selesai diproses.\n")

# ============================================================================
# 5. SIMPAN HASIL KE CSV
# ============================================================================
df_params = pd.DataFrame(ets_params)
params_file = os.path.join(OUTPUT_TABLES, "ets_parameters.csv")
df_params.to_csv(params_file, index=False)
print(f"[OK] Parameter ETS     : {params_file}")

df_eval = pd.DataFrame(eval_metrics)
eval_file = os.path.join(OUTPUT_TABLES, "ets_evaluation.csv")
df_eval.to_csv(eval_file, index=False)
print(f"[OK] Evaluasi ETS      : {eval_file}")

if len(forecast_records) > 0:
    df_forecast = pd.DataFrame(forecast_records)
    df_forecast['date'] = pd.to_datetime(df_forecast['date'])
else:
    df_forecast = pd.DataFrame(columns=['date','commodity','forecast','lower_95','upper_95','model'])

forecast_file = os.path.join(OUTPUT_TABLES, "ets_forecast_2026_2030.csv")
df_forecast.to_csv(forecast_file, index=False)
print(f"[OK] Forecast 2026-2030: {forecast_file}")

# ============================================================================
# 6. VISUALISASI  -  FORECAST vs ACTUAL PER KOMODITAS
# ============================================================================
print("\n[>] Membuat visualisasi forecast vs actual...")

n_cols = 2
n_rows = (len(COMMODITIES) + 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for idx, commodity in enumerate(COMMODITIES):
    ax = axes[idx]
    hist = df[commodity]

    fc_df = df_forecast[df_forecast['commodity'] == commodity].copy().sort_values('date')

    # Historis
    ax.plot(hist.index, hist.values, color='#2C3E50', linewidth=1.5,
            label='Historical', alpha=0.9)

    # Test-set prediction (dari model training)
    result_tr = ets_results_train.get(commodity)
    test_s    = df_test[commodity].dropna()
    if result_tr is not None and len(test_s) > 0:
        try:
            pred_test = result_tr.forecast(len(test_s))
            pred_test.index = test_s.index
            pred_test = pred_test.clip(lower=0)
            ax.plot(pred_test.index, pred_test.values,
                    color='#F39C12', linewidth=1.5, linestyle='--',
                    label='Pred. (Test)', alpha=0.85)
        except Exception:
            pass

    # Forecast masa depan
    if len(fc_df) > 0:
        ax.plot(fc_df['date'], fc_df['forecast'],
                color='#8E44AD', linewidth=2, label='Forecast ETS 2026-2030')
        ax.fill_between(
            fc_df['date'], fc_df['lower_95'], fc_df['upper_95'],
            color='#8E44AD', alpha=0.15, label='CI 95%'
        )

    ax.axvline(pd.Timestamp(TRAIN_END), color='gray', linestyle=':', linewidth=1,
               label='Train/Test split')
    ax.set_title(f'{commodity}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Harga (IDR)', fontsize=9)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True, alpha=0.3)

for j in range(len(COMMODITIES), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('ETS (Holt-Winters)  -  Forecast vs Actual (2007-2030)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig_file = os.path.join(OUTPUT_FIGURES, "14_ets_forecast_vs_actual.png")
plt.savefig(fig_file, dpi=150, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: {fig_file}")

# ============================================================================
# 7. VISUALISASI  -  FORECAST SHORT-TERM (ZOOM 2020-2030)
# ============================================================================
print("[>] Membuat visualisasi forecast short-term (zoom 2020-2030)...")

fig2, axes2 = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes2 = axes2.flatten()

for idx, commodity in enumerate(COMMODITIES):
    ax = axes2[idx]
    hist = df[commodity].loc['2020-01-01':]
    fc_df = df_forecast[df_forecast['commodity'] == commodity].copy().sort_values('date')

    ax.plot(hist.index, hist.values, color='#2C3E50', linewidth=2,
            label='Historis', alpha=0.9)

    if len(fc_df) > 0:
        ax.plot(fc_df['date'], fc_df['forecast'],
                color='#8E44AD', linewidth=2.5, label='Forecast ETS')
        ax.fill_between(
            fc_df['date'], fc_df['lower_95'], fc_df['upper_95'],
            color='#8E44AD', alpha=0.2, label='CI 95%'
        )

    ax.axvline(LAST_DATA_DATE, color='#27AE60', linestyle='--', linewidth=1.5,
               label='Batas data')
    ax.set_title(f'{commodity}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Harga (IDR)', fontsize=9)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True, alpha=0.3)

for j in range(len(COMMODITIES), len(axes2)):
    fig2.delaxes(axes2[j])

plt.suptitle('ETS (Holt-Winters)  -  Forecast Harga Pangan 2026-2030\n(Zoom 2020-2030)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig2_file = os.path.join(OUTPUT_FIGURES, "15_ets_forecast_2026_2030.png")
plt.savefig(fig2_file, dpi=150, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: {fig2_file}")

# ============================================================================
# 8. VISUALISASI  -  SARIMA vs ETS MAPE COMPARISON
# ============================================================================
print("[>] Membuat bar chart perbandingan MAPE SARIMA vs ETS...")

sarima_eval_file = os.path.join(OUTPUT_TABLES, "sarima_evaluation.csv")
if os.path.exists(sarima_eval_file):
    df_sarima_eval = pd.read_csv(sarima_eval_file)

    df_compare = pd.merge(
        df_sarima_eval[['Commodity', 'MAPE_%']].rename(columns={'MAPE_%': 'SARIMA'}),
        df_eval[['Commodity', 'MAPE_%']].rename(columns={'MAPE_%': 'ETS'}),
        on='Commodity', how='outer'
    ).sort_values('ETS')

    fig3, ax3 = plt.subplots(figsize=(12, 7))
    x = np.arange(len(df_compare))
    w = 0.35

    bars1 = ax3.barh(x + w/2, df_compare['SARIMA'], w, label='SARIMA (log)',
                     color='#E74C3C', alpha=0.85, edgecolor='white')
    bars2 = ax3.barh(x - w/2, df_compare['ETS'],    w, label='ETS (Holt-Winters)',
                     color='#8E44AD', alpha=0.85, edgecolor='white')

    ax3.set_yticks(x)
    ax3.set_yticklabels(df_compare['Commodity'], fontsize=10)
    ax3.set_xlabel('MAPE (%)', fontsize=12)
    ax3.set_title('Perbandingan MAPE: SARIMA vs ETS\n(Test Set Jan 2022 - Mei 2025)',
                  fontsize=13, fontweight='bold')
    ax3.legend(fontsize=11)
    ax3.grid(True, axis='x', alpha=0.3)
    ax3.axvline(10, color='gray', linestyle='--', linewidth=1, alpha=0.6)
    ax3.text(10.2, -0.8, '10% threshold', fontsize=9, color='gray')

    # Label nilai
    for bar in bars1:
        w_ = bar.get_width()
        if not np.isnan(w_):
            ax3.text(w_ + 0.3, bar.get_y() + bar.get_height()/2,
                     f'{w_:.1f}%', va='center', fontsize=8, color='#E74C3C')
    for bar in bars2:
        w_ = bar.get_width()
        if not np.isnan(w_):
            ax3.text(w_ + 0.3, bar.get_y() + bar.get_height()/2,
                     f'{w_:.1f}%', va='center', fontsize=8, color='#8E44AD')

    plt.tight_layout()
    fig3_file = os.path.join(OUTPUT_FIGURES, "16_sarima_vs_ets_mape.png")
    plt.savefig(fig3_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   [OK] Tersimpan: {fig3_file}")
else:
    print("   [SKIP] sarima_evaluation.csv tidak ditemukan, skip comparison chart.")

# ============================================================================
# 9. PRINT SUMMARY AKHIR
# ============================================================================
print("\n" + "=" * 80)
print("RINGKASAN HASIL ETS")
print("=" * 80)

print("\n--- KONFIGURASI ETS TERBAIK ---")
print(df_params.to_string(index=False))

print("\n--- EVALUASI (Test Set, diurutkan MAPE) ---")
print(df_eval.sort_values('MAPE_%').to_string(index=False))

# Preview forecast Des 2030
dec2030 = df_forecast[
    df_forecast['date'].astype(str).str.startswith('2030-12')
][['commodity', 'forecast', 'lower_95', 'upper_95']].sort_values('commodity')

if len(dec2030) > 0:
    print("\n--- PROYEKSI HARGA DESEMBER 2030 (IDR) ---")
    for _, row in dec2030.iterrows():
        print(f"  {row['commodity']:<30} {row['forecast']:>12,.0f}  "
              f"(CI: {row['lower_95']:,.0f} - {row['upper_95']:,.0f})")

# Bandingkan MAPE dengan SARIMA
if os.path.exists(sarima_eval_file):
    df_sarima_eval = pd.read_csv(sarima_eval_file)
    df_compare_final = pd.merge(
        df_sarima_eval[['Commodity', 'MAPE_%']].rename(columns={'MAPE_%': 'MAPE_SARIMA'}),
        df_eval[['Commodity', 'MAPE_%']].rename(columns={'MAPE_%': 'MAPE_ETS'}),
        on='Commodity', how='outer'
    )
    df_compare_final['winner'] = df_compare_final.apply(
        lambda r: 'ETS' if r['MAPE_ETS'] < r['MAPE_SARIMA'] else 'SARIMA', axis=1
    )
    df_compare_final['delta_%'] = (df_compare_final['MAPE_ETS'] - df_compare_final['MAPE_SARIMA']).round(2)
    print("\n--- SARIMA vs ETS: MODEL LEBIH BAIK PER KOMODITAS ---")
    print(df_compare_final.sort_values('MAPE_ETS').to_string(index=False))

    n_ets_wins   = (df_compare_final['winner'] == 'ETS').sum()
    n_sarima_wins = (df_compare_final['winner'] == 'SARIMA').sum()
    print(f"\n  ETS menang   : {n_ets_wins} komoditas")
    print(f"  SARIMA menang: {n_sarima_wins} komoditas")

print("""
[DONE] PHASE 5 COMPLETE
   Files tersimpan:
     - outputs/tables/ets_parameters.csv
     - outputs/tables/ets_evaluation.csv
     - outputs/tables/ets_forecast_2026_2030.csv
     - outputs/figures/14_ets_forecast_vs_actual.png
     - outputs/figures/15_ets_forecast_2026_2030.png
     - outputs/figures/16_sarima_vs_ets_mape.png

   Next Step --> Phase 6: Prophet Modeling
""")


## Tahap: 06_prophet_model.py

In [ ]:
"""
PHASE 6: PROPHET MODELING
Facebook/Meta Prophet per komoditas

Kelebihan Prophet vs SARIMA/ETS:
- Robust terhadap missing values & outlier
- Menangkap tren non-linear secara otomatis (changepoints)
- Seasonal additive: yearly + weekly + custom (Ramadan)
- Tidak perlu stationarity  -  bekerja pada harga asli (level)
- Uncertainty interval dari posterior sampling

Strategi:
- Fit pada training (Jan 2007 - Des 2021), raw IDR (tanpa transform)
- Tambahkan seasonality Ramadan/Lebaran sebagai custom regressor
- Evaluasi pada test set (Jan 2022 - Mei 2025)
- Forecast: Jun 2025 - Des 2030 (67 bulan)
- Bandingkan MAPE dengan SARIMA dan ETS
"""

import pandas as pd
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')   # non-interactive backend agar tidak pop-up window
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import logging

warnings.filterwarnings('ignore')
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ============================================================================
# SETUP PATH PORTABLE
# ============================================================================
def _resolve_project_dir():
    """Ambil root proyek; kalau dijalankan dari folder Code, naik ke folder induk."""
    env_dir = os.environ.get("PROJECT_DIR")
    if env_dir:
        return os.path.abspath(env_dir)

    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()

    base_dir = os.path.abspath(base_dir)
    parent_dir = os.path.abspath(os.path.join(base_dir, os.pardir))

    # Struktur proyek: <PROJECT_DIR>/Code/code.ipynb, <PROJECT_DIR>/dataset, <PROJECT_DIR>/outputs
    # Jadi saat notebook/script dijalankan dari folder Code, output tetap ke folder outputs di root proyek.
    if os.path.basename(base_dir).lower() == "code":
        return parent_dir

    if os.path.isdir(os.path.join(base_dir, "dataset")):
        return base_dir
    if os.path.isdir(os.path.join(parent_dir, "dataset")):
        return parent_dir

    return base_dir

PROJECT_DIR = _resolve_project_dir()
OUTPUT_DATA    = os.path.join(PROJECT_DIR, "outputs")
OUTPUT_FIGURES = os.path.join(OUTPUT_DATA, "figures")
OUTPUT_TABLES  = os.path.join(OUTPUT_DATA, "tables")

os.makedirs(OUTPUT_DATA, exist_ok=True)
os.makedirs(OUTPUT_FIGURES, exist_ok=True)
os.makedirs(OUTPUT_TABLES,  exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150

print("=" * 80)
print("PHASE 6: PROPHET MODELING")
print("=" * 80)

# ============================================================================
# 1. LOAD DATA CLEAN
# ============================================================================
input_file = os.path.join(OUTPUT_DATA, "national_avg_clean_fixed.csv")
df = pd.read_csv(input_file, parse_dates=['date'], index_col='date')

df.index = pd.to_datetime(df.index).to_period('M').to_timestamp()
df = df.sort_index()
full_idx = pd.date_range(df.index[0], df.index[-1], freq='MS')
df = df.reindex(full_idx).ffill()

print(f"\n[OK] Data dimuat: {df.shape}")
print(f"     Rentang: {df.index[0].strftime('%b %Y')} - {df.index[-1].strftime('%b %Y')}")
print(f"     Komoditas: {list(df.columns)}\n")

COMMODITIES = list(df.columns)

# ============================================================================
# 2. TRAIN / TEST SPLIT
# ============================================================================
TRAIN_END      = '2021-12-01'
TEST_START     = '2022-01-01'
TEST_END       = df.index[-1]
LAST_DATA_DATE = df.index[-1]
FORECAST_END   = pd.Timestamp('2030-12-01')
FORECAST_STEPS = len(pd.date_range(LAST_DATA_DATE, FORECAST_END, freq='MS')) - 1

print(f"[INFO] Train   : Jan 2007 - {TRAIN_END}")
print(f"[INFO] Test    : {TEST_START} - {TEST_END.strftime('%Y-%m-%d')}")
print(f"[INFO] Forecast: {(LAST_DATA_DATE + pd.DateOffset(months=1)).strftime('%b %Y')} - Des 2030")
print(f"       Steps forecast: {FORECAST_STEPS} bulan\n")

df_train = df.loc[:TRAIN_END]
df_test  = df.loc[TEST_START:TEST_END]

# ============================================================================
# 3. FUNGSI UTILITAS
# ============================================================================

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def make_prophet_df(series):
    """Konversi pd.Series ke format Prophet: columns [ds, y]."""
    return pd.DataFrame({'ds': series.index, 'y': series.values}).reset_index(drop=True)


def get_ramadan_holidays(start_year=2007, end_year=2031):
    """
    Buat DataFrame holiday Prophet untuk Ramadan & Lebaran.
    Tanggal Ramadan (approx) per tahun  -  1 bulan sebelum Idul Fitri.
    """
    # Perkiraan tanggal Idul Fitri (awal bulan)
    eid_dates = {
        2007: '2007-10-13', 2008: '2008-10-01', 2009: '2009-09-20',
        2010: '2010-09-10', 2011: '2011-08-30', 2012: '2012-08-19',
        2013: '2013-08-08', 2014: '2014-07-28', 2015: '2015-07-17',
        2016: '2016-07-06', 2017: '2017-06-25', 2018: '2018-06-15',
        2019: '2019-06-05', 2020: '2020-05-24', 2021: '2021-05-13',
        2022: '2022-05-02', 2023: '2023-04-22', 2024: '2024-04-10',
        2025: '2025-03-30', 2026: '2026-03-20', 2027: '2027-03-09',
        2028: '2028-02-26', 2029: '2029-02-14', 2030: '2030-02-04',
        2031: '2031-01-24',
    }

    rows = []
    for year, eid_str in eid_dates.items():
        if year < start_year or year > end_year:
            continue
        eid = pd.Timestamp(eid_str)
        # Ramadan: 30 hari sebelum Idul Fitri
        ramadan_start = eid - pd.DateOffset(days=30)

        rows.append({'holiday': 'ramadan',    'ds': ramadan_start,
                     'lower_window': 0, 'upper_window': 29})
        rows.append({'holiday': 'lebaran',    'ds': eid,
                     'lower_window': -1, 'upper_window': 6})

    return pd.DataFrame(rows)


holidays_df = get_ramadan_holidays()

# ============================================================================
# 4. PROPHET FITTING PER KOMODITAS
# ============================================================================
print("=" * 80)
print("PROPHET FITTING PER KOMODITAS")
print("=" * 80)

prophet_results   = {}   # {commodity: model fitted on train}
prophet_params    = []
eval_metrics      = []
forecast_records  = []

for commodity in COMMODITIES:
    print(f"\n[>] {commodity}")

    train_series = df_train[commodity].dropna()
    test_series  = df_test[commodity].dropna()

    train_prophet = make_prophet_df(train_series)

    # --- Fit Prophet pada training set ---
    try:
        model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
            seasonality_mode='multiplicative',
            changepoint_prior_scale=0.05,
            seasonality_prior_scale=10,
            holidays=holidays_df,
            holidays_prior_scale=10,
        )
        model.fit(train_prophet)
        print(f"    Fit selesai (multiplicative, Ramadan holiday, changepoints)")
    except Exception as e:
        print(f"    [WARN] Fit multiplicative gagal, coba additive...")
        try:
            model = Prophet(
                yearly_seasonality=True,
                weekly_seasonality=False,
                daily_seasonality=False,
                seasonality_mode='additive',
                changepoint_prior_scale=0.05,
                holidays=holidays_df,
            )
            model.fit(train_prophet)
            print(f"    Fit selesai (additive)")
        except Exception as e2:
            print(f"    [ERROR] Kedua mode gagal: {e2}. Skip.")
            continue

    prophet_results[commodity] = model

    # Simpan parameter
    prophet_params.append({
        'Commodity'              : commodity,
        'seasonality_mode'       : model.seasonality_mode,
        'changepoint_prior_scale': model.changepoint_prior_scale,
        'yearly_seasonality'     : True,
        'holidays'               : 'Ramadan+Lebaran',
        'n_changepoints'         : model.n_changepoints,
    })

    # --- Evaluasi pada test set ---
    n_test = len(test_series)
    if n_test > 0:
        try:
            future_test = model.make_future_dataframe(
                periods=n_test, freq='MS', include_history=False
            )
            forecast_test = model.predict(future_test)
            pred_vals = forecast_test['yhat'].values
            pred_vals = np.clip(pred_vals, 0, None)   # clip negatif

            actual_vals = test_series.values[:len(pred_vals)]

            mae_val  = mean_absolute_error(actual_vals, pred_vals)
            rmse_val = np.sqrt(mean_squared_error(actual_vals, pred_vals))
            mape_val = mape(actual_vals, pred_vals)

            print(f"    MAE={mae_val:,.0f}  RMSE={rmse_val:,.0f}  MAPE={mape_val:.2f}%")

            eval_metrics.append({
                'Commodity': commodity,
                'Model'    : 'Prophet',
                'MAE'      : round(mae_val, 2),
                'RMSE'     : round(rmse_val, 2),
                'MAPE_%'   : round(mape_val, 2),
                'n_test'   : n_test,
            })
        except Exception as e:
            print(f"    [WARN] Evaluasi test set gagal: {e}")
            eval_metrics.append({
                'Commodity': commodity, 'Model': 'Prophet',
                'MAE': np.nan, 'RMSE': np.nan, 'MAPE_%': np.nan, 'n_test': n_test,
            })
    else:
        print("    [WARN] Test set kosong.")

    # --- Re-fit pada SELURUH data (untuk forecast masa depan) ---
    full_series  = df[commodity].dropna()
    full_prophet = make_prophet_df(full_series)

    try:
        model_full = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
            seasonality_mode=model.seasonality_mode,
            changepoint_prior_scale=0.05,
            seasonality_prior_scale=10,
            holidays=holidays_df,
            holidays_prior_scale=10,
        )
        model_full.fit(full_prophet)
    except Exception:
        model_full = model   # fallback ke model training

    # --- Forecast masa depan ---
    if FORECAST_STEPS > 0:
        try:
            future_fc = model_full.make_future_dataframe(
                periods=FORECAST_STEPS, freq='MS', include_history=False
            )
            fc = model_full.predict(future_fc)

            fc_dates = pd.date_range(
                start=LAST_DATA_DATE + pd.DateOffset(months=1),
                periods=FORECAST_STEPS,
                freq='MS'
            )

            for i in range(min(FORECAST_STEPS, len(fc))):
                fc_val   = max(float(fc['yhat'].iloc[i]), 0)
                fc_lower = max(float(fc['yhat_lower'].iloc[i]), 0)
                fc_upper = max(float(fc['yhat_upper'].iloc[i]), 0)

                forecast_records.append({
                    'date'     : fc_dates[i],
                    'commodity': commodity,
                    'forecast' : round(fc_val, 2),
                    'lower_95' : round(fc_lower, 2),
                    'upper_95' : round(fc_upper, 2),
                    'model'    : 'Prophet',
                })
        except Exception as e:
            print(f"    [WARN] Forecast gagal: {e}")

print("\n[OK] Semua komoditas selesai diproses.\n")

# ============================================================================
# 5. SIMPAN HASIL KE CSV
# ============================================================================
df_params = pd.DataFrame(prophet_params)
params_file = os.path.join(OUTPUT_TABLES, "prophet_parameters.csv")
df_params.to_csv(params_file, index=False)
print(f"[OK] Parameter Prophet  : {params_file}")

df_eval = pd.DataFrame(eval_metrics) if eval_metrics else pd.DataFrame(
    columns=['Commodity','Model','MAE','RMSE','MAPE_%','n_test'])
eval_file = os.path.join(OUTPUT_TABLES, "prophet_evaluation.csv")
df_eval.to_csv(eval_file, index=False)
print(f"[OK] Evaluasi Prophet   : {eval_file}")

if len(forecast_records) > 0:
    df_forecast = pd.DataFrame(forecast_records)
    df_forecast['date'] = pd.to_datetime(df_forecast['date'])
else:
    df_forecast = pd.DataFrame(
        columns=['date','commodity','forecast','lower_95','upper_95','model'])

forecast_file = os.path.join(OUTPUT_TABLES, "prophet_forecast_2026_2030.csv")
df_forecast.to_csv(forecast_file, index=False)
print(f"[OK] Forecast 2026-2030 : {forecast_file}")

# ============================================================================
# 6. VISUALISASI  -  FORECAST vs ACTUAL PER KOMODITAS
# ============================================================================
print("\n[>] Membuat visualisasi forecast vs actual...")

n_cols = 2
n_rows = (len(COMMODITIES) + 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for idx, commodity in enumerate(COMMODITIES):
    ax = axes[idx]
    hist = df[commodity]
    fc_df = df_forecast[df_forecast['commodity'] == commodity].copy().sort_values('date')

    ax.plot(hist.index, hist.values, color='#2C3E50', linewidth=1.5,
            label='Historical', alpha=0.9)

    # Test-set prediction
    model_tr = prophet_results.get(commodity)
    test_s   = df_test[commodity].dropna()
    if model_tr is not None and len(test_s) > 0:
        try:
            fut_t = model_tr.make_future_dataframe(
                periods=len(test_s), freq='MS', include_history=False)
            fc_t = model_tr.predict(fut_t)
            pred_vals = np.clip(fc_t['yhat'].values, 0, None)
            ax.plot(test_s.index[:len(pred_vals)], pred_vals,
                    color='#F39C12', linewidth=1.5, linestyle='--',
                    label='Pred. (Test)', alpha=0.85)
        except Exception:
            pass

    # Forecast
    if len(fc_df) > 0:
        ax.plot(fc_df['date'], fc_df['forecast'],
                color='#27AE60', linewidth=2, label='Forecast Prophet')
        ax.fill_between(
            fc_df['date'], fc_df['lower_95'], fc_df['upper_95'],
            color='#27AE60', alpha=0.15, label='CI 95%'
        )

    ax.axvline(pd.Timestamp(TRAIN_END), color='gray',
               linestyle=':', linewidth=1, label='Train/Test split')
    ax.set_title(f'{commodity}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Harga (IDR)', fontsize=9)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True, alpha=0.3)

for j in range(len(COMMODITIES), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Prophet  -  Forecast vs Actual (2007-2030)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig_file = os.path.join(OUTPUT_FIGURES, "17_prophet_forecast_vs_actual.png")
plt.savefig(fig_file, dpi=150, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: {fig_file}")

# ============================================================================
# 7. VISUALISASI  -  FORECAST SHORT-TERM (ZOOM 2020-2030)
# ============================================================================
print("[>] Membuat visualisasi forecast short-term (zoom 2020-2030)...")

fig2, axes2 = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes2 = axes2.flatten()

for idx, commodity in enumerate(COMMODITIES):
    ax = axes2[idx]
    hist = df[commodity].loc['2020-01-01':]
    fc_df = df_forecast[df_forecast['commodity'] == commodity].copy().sort_values('date')

    ax.plot(hist.index, hist.values, color='#2C3E50',
            linewidth=2, label='Historis', alpha=0.9)

    if len(fc_df) > 0:
        ax.plot(fc_df['date'], fc_df['forecast'],
                color='#27AE60', linewidth=2.5, label='Forecast Prophet')
        ax.fill_between(
            fc_df['date'], fc_df['lower_95'], fc_df['upper_95'],
            color='#27AE60', alpha=0.2, label='CI 95%'
        )

    ax.axvline(LAST_DATA_DATE, color='#E74C3C',
               linestyle='--', linewidth=1.5, label='Batas data')
    ax.set_title(f'{commodity}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Harga (IDR)', fontsize=9)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True, alpha=0.3)

for j in range(len(COMMODITIES), len(axes2)):
    fig2.delaxes(axes2[j])

plt.suptitle('Prophet  -  Forecast Harga Pangan 2026-2030\n(Zoom 2020-2030)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig2_file = os.path.join(OUTPUT_FIGURES, "18_prophet_forecast_2026_2030.png")
plt.savefig(fig2_file, dpi=150, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: {fig2_file}")

# ============================================================================
# 8. VISUALISASI  -  3-WAY MAPE COMPARISON (SARIMA vs ETS vs Prophet)
# ============================================================================
print("[>] Membuat 3-way MAPE comparison chart...")

sarima_file = os.path.join(OUTPUT_TABLES, "sarima_evaluation.csv")
ets_file    = os.path.join(OUTPUT_TABLES, "ets_evaluation.csv")

if os.path.exists(sarima_file) and os.path.exists(ets_file) and not df_eval.empty:
    df_s = pd.read_csv(sarima_file)[['Commodity','MAPE_%']].rename(
        columns={'MAPE_%': 'SARIMA'})
    df_e = pd.read_csv(ets_file)[['Commodity','MAPE_%']].rename(
        columns={'MAPE_%': 'ETS'})
    df_p = df_eval[['Commodity','MAPE_%']].rename(
        columns={'MAPE_%': 'Prophet'})

    df_3way = df_s.merge(df_e, on='Commodity').merge(df_p, on='Commodity')
    df_3way['Best'] = df_3way[['SARIMA','ETS','Prophet']].idxmin(axis=1)
    df_3way = df_3way.sort_values('Prophet')

    fig3, ax3 = plt.subplots(figsize=(13, 8))
    x = np.arange(len(df_3way))
    w = 0.25

    ax3.barh(x + w,   df_3way['SARIMA'],  w, label='SARIMA (log)',
             color='#E74C3C', alpha=0.85, edgecolor='white')
    ax3.barh(x,       df_3way['ETS'],     w, label='ETS (Holt-Winters)',
             color='#8E44AD', alpha=0.85, edgecolor='white')
    ax3.barh(x - w,   df_3way['Prophet'], w, label='Prophet',
             color='#27AE60', alpha=0.85, edgecolor='white')

    ax3.set_yticks(x)
    ax3.set_yticklabels(df_3way['Commodity'], fontsize=10)
    ax3.set_xlabel('MAPE (%)', fontsize=12)
    ax3.set_title('Perbandingan MAPE: SARIMA vs ETS vs Prophet\n(Test Set Jan 2022 - Mei 2025)',
                  fontsize=13, fontweight='bold')
    ax3.legend(fontsize=11, loc='lower right')
    ax3.grid(True, axis='x', alpha=0.3)
    ax3.axvline(10, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax3.text(10.2, -1, '10% threshold', fontsize=8, color='gray')

    plt.tight_layout()
    fig3_file = os.path.join(OUTPUT_FIGURES, "19_three_model_mape_comparison.png")
    plt.savefig(fig3_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   [OK] Tersimpan: {fig3_file}")

# ============================================================================
# 9. PRINT SUMMARY AKHIR
# ============================================================================
print("\n" + "=" * 80)
print("RINGKASAN HASIL PROPHET")
print("=" * 80)

print("\n--- KONFIGURASI PROPHET ---")
print(df_params.to_string(index=False))

print("\n--- EVALUASI (Test Set, diurutkan MAPE) ---")
print(df_eval.sort_values('MAPE_%').to_string(index=False))

# Preview Des 2030
dec2030 = df_forecast[
    df_forecast['date'].astype(str).str.startswith('2030-12')
][['commodity','forecast','lower_95','upper_95']].sort_values('commodity')

if len(dec2030) > 0:
    print("\n--- PROYEKSI HARGA DESEMBER 2030 (IDR) ---")
    for _, row in dec2030.iterrows():
        print(f"  {row['commodity']:<30} {row['forecast']:>12,.0f}  "
              f"(CI: {row['lower_95']:,.0f} - {row['upper_95']:,.0f})")

# 3-way comparison summary
if os.path.exists(sarima_file) and os.path.exists(ets_file) and not df_eval.empty:
    print("\n--- 3-WAY MODEL COMPARISON ---")
    print(f"  {'Commodity':<30} {'SARIMA':>8} {'ETS':>8} {'Prophet':>8} {'BEST':>8}")
    print(f"  {'-'*62}")
    for _, r in df_3way.sort_values('Prophet').iterrows():
        print(f"  {r['Commodity']:<30} {r['SARIMA']:>8.2f} {r['ETS']:>8.2f} "
              f"{r['Prophet']:>8.2f} {r['Best']:>8}")

    wins = df_3way['Best'].value_counts()
    print(f"\n  SARIMA menang : {wins.get('SARIMA', 0)} komoditas")
    print(f"  ETS menang    : {wins.get('ETS', 0)} komoditas")
    print(f"  Prophet menang: {wins.get('Prophet', 0)} komoditas")

print("""
[DONE] PHASE 6 COMPLETE
   Files tersimpan:
     - outputs/tables/prophet_parameters.csv
     - outputs/tables/prophet_evaluation.csv
     - outputs/tables/prophet_forecast_2026_2030.csv
     - outputs/figures/17_prophet_forecast_vs_actual.png
     - outputs/figures/18_prophet_forecast_2026_2030.png
     - outputs/figures/19_three_model_mape_comparison.png

   Next Step --> Phase 7: Model Comparison & Selection
""")


## Tahap: 07_model_comparison.py

In [ ]:
"""
PHASE 7: MODEL COMPARISON & SELECTION
Pilih model terbaik per komoditas berdasarkan MAPE test set,
lalu gabungkan forecast terbaik ke satu tabel final.

Output:
- best_model_per_commodity.csv   -  model winner per komoditas
- final_forecast_2026_2030.csv   -  forecast terbaik per komoditas
- final_forecast_wide.csv        -  format wide (pivot per komoditas)
- 20_best_model_summary.png      -  heatmap & bar chart model winner
- 21_final_forecast_all.png      -  forecast terbaik semua komoditas
- 22_model_comparison_radar.png  -  radar chart perbandingan 3 model
"""

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ============================================================================
# SETUP PATH PORTABLE
# ============================================================================
def _resolve_project_dir():
    """Ambil root proyek; kalau dijalankan dari folder Code, naik ke folder induk."""
    env_dir = os.environ.get("PROJECT_DIR")
    if env_dir:
        return os.path.abspath(env_dir)

    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()

    base_dir = os.path.abspath(base_dir)
    parent_dir = os.path.abspath(os.path.join(base_dir, os.pardir))

    # Struktur proyek: <PROJECT_DIR>/Code/code.ipynb, <PROJECT_DIR>/dataset, <PROJECT_DIR>/outputs
    # Jadi saat notebook/script dijalankan dari folder Code, output tetap ke folder outputs di root proyek.
    if os.path.basename(base_dir).lower() == "code":
        return parent_dir

    if os.path.isdir(os.path.join(base_dir, "dataset")):
        return base_dir
    if os.path.isdir(os.path.join(parent_dir, "dataset")):
        return parent_dir

    return base_dir

PROJECT_DIR = _resolve_project_dir()
OUTPUT_DATA    = os.path.join(PROJECT_DIR, "outputs")
OUTPUT_FIGURES = os.path.join(OUTPUT_DATA, "figures")
OUTPUT_TABLES  = os.path.join(OUTPUT_DATA, "tables")

os.makedirs(OUTPUT_DATA, exist_ok=True)
os.makedirs(OUTPUT_FIGURES, exist_ok=True)
os.makedirs(OUTPUT_TABLES,  exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150

print("=" * 80)
print("PHASE 7: MODEL COMPARISON & SELECTION")
print("=" * 80)

# ============================================================================
# 1. LOAD SEMUA HASIL EVALUASI
# ============================================================================
df_sarima = pd.read_csv(os.path.join(OUTPUT_TABLES, "sarima_evaluation.csv"))
df_ets    = pd.read_csv(os.path.join(OUTPUT_TABLES, "ets_evaluation.csv"))
df_prophet= pd.read_csv(os.path.join(OUTPUT_TABLES, "prophet_evaluation.csv"))

# Load semua forecast
fc_sarima  = pd.read_csv(os.path.join(OUTPUT_TABLES, "sarima_forecast_2026_2030.csv"))
fc_ets     = pd.read_csv(os.path.join(OUTPUT_TABLES, "ets_forecast_2026_2030.csv"))
fc_prophet = pd.read_csv(os.path.join(OUTPUT_TABLES, "prophet_forecast_2026_2030.csv"))

for fc in [fc_sarima, fc_ets, fc_prophet]:
    fc['date'] = pd.to_datetime(fc['date'])

# Load historis
df_hist = pd.read_csv(
    os.path.join(OUTPUT_DATA, "national_avg_clean_fixed.csv"),
    parse_dates=['date'], index_col='date'
)
df_hist.index = pd.to_datetime(df_hist.index).to_period('M').to_timestamp()

print(f"\n[OK] Evaluasi dimuat:")
print(f"     SARIMA  : {len(df_sarima)} komoditas")
print(f"     ETS     : {len(df_ets)} komoditas")
print(f"     Prophet : {len(df_prophet)} komoditas")

# ============================================================================
# 2. MERGE EVALUASI  -  PILIH MODEL TERBAIK
# ============================================================================
df_cmp = (
    df_sarima[['Commodity','MAE','RMSE','MAPE_%']]
        .rename(columns={'MAE':'MAE_SARIMA','RMSE':'RMSE_SARIMA','MAPE_%':'MAPE_SARIMA'})
    .merge(
        df_ets[['Commodity','MAE','RMSE','MAPE_%']]
            .rename(columns={'MAE':'MAE_ETS','RMSE':'RMSE_ETS','MAPE_%':'MAPE_ETS'}),
        on='Commodity', how='outer'
    )
    .merge(
        df_prophet[['Commodity','MAE','RMSE','MAPE_%']]
            .rename(columns={'MAE':'MAE_Prophet','RMSE':'RMSE_Prophet','MAPE_%':'MAPE_Prophet'}),
        on='Commodity', how='outer'
    )
)

# Pilih model dengan MAPE terendah  -  handle NaN dengan fill inf
def pick_best(row):
    candidates = {
        'SARIMA' : row.get('MAPE_SARIMA',  np.inf) or np.inf,
        'ETS'    : row.get('MAPE_ETS',     np.inf) or np.inf,
        'Prophet': row.get('MAPE_Prophet', np.inf) or np.inf,
    }
    best = min(candidates, key=candidates.get)
    return best, round(candidates[best], 2)

df_cmp['Best_Model'], df_cmp['Best_MAPE'] = zip(*df_cmp.apply(pick_best, axis=1))

# ============================================================================
# MANUAL OVERRIDES  -  berdasarkan validasi kualitatif forecast jangka panjang
# ============================================================================
OVERRIDES = {
    # SARIMA Eggs collapse ke Rp 212 di 2030 akibat level-shift data enrichment.
    # ETS menghasilkan Rp 14.869 (masuk akal, mendekati harga enrichment terakhir).
    'Eggs': ('ETS', 'SARIMA forecast collapse (Rp 212) akibat level-shift data enrichment 2024'),

    # ETS Meat (chicken) memilih trend=None -> forecast konstan Rp 40.365 sepanjang 2026-2030.
    # SARIMA menghasilkan tren naik gradual ke Rp 43.525 (lebih informatif secara ekonomis).
    'Meat (chicken, broiler)': ('SARIMA', 'ETS flat forecast (trend=None), SARIMA lebih informatif'),
}

print("\n[INFO] MANUAL OVERRIDES:")
for comm, (new_model, reason) in OVERRIDES.items():
    old_model = df_cmp.loc[df_cmp['Commodity'] == comm, 'Best_Model'].values[0]
    df_cmp.loc[df_cmp['Commodity'] == comm, 'Best_Model'] = new_model
    # Update Best_MAPE to match the overridden model's MAPE
    mape_col = f'MAPE_{new_model}'
    new_mape = df_cmp.loc[df_cmp['Commodity'] == comm, mape_col].values[0]
    df_cmp.loc[df_cmp['Commodity'] == comm, 'Best_MAPE'] = round(new_mape, 2)
    print(f"  {comm}: {old_model} -> {new_model}")
    print(f"    Alasan: {reason}")
    print(f"    MAPE updated: {df_cmp.loc[df_cmp['Commodity'] == comm, 'Best_MAPE'].values[0]:.2f}%")

# Rank masing-masing model
df_cmp = df_cmp.sort_values('Best_MAPE')

print("\n" + "=" * 80)
print("HASIL PERBANDINGAN MODEL")
print("=" * 80)
print(f"\n{'Commodity':<30} {'SARIMA%':>8} {'ETS%':>8} {'Prophet%':>8} {'BEST':>10} {'MAPE%':>7}")
print("-" * 75)
for _, r in df_cmp.iterrows():
    marker_s = '*' if r['Best_Model'] == 'SARIMA'  else ' '
    marker_e = '*' if r['Best_Model'] == 'ETS'     else ' '
    marker_p = '*' if r['Best_Model'] == 'Prophet' else ' '
    print(f"  {r['Commodity']:<28} "
          f"{str(round(r['MAPE_SARIMA'],2))+'%'+marker_s:>9} "
          f"{str(round(r['MAPE_ETS'],2))+'%'+marker_e:>9} "
          f"{str(round(r['MAPE_Prophet'],2))+'%'+marker_p:>10}  "
          f"{r['Best_Model']:>8}  {r['Best_MAPE']:>6.2f}%")

wins = df_cmp['Best_Model'].value_counts()
print(f"\n  SARIMA  menang: {wins.get('SARIMA',0)} komoditas")
print(f"  ETS     menang: {wins.get('ETS',0)} komoditas")
print(f"  Prophet menang: {wins.get('Prophet',0)} komoditas")

# Simpan tabel perbandingan
df_cmp.to_csv(os.path.join(OUTPUT_TABLES, "best_model_per_commodity.csv"), index=False)
print(f"\n[OK] Tabel comparison: {os.path.join(OUTPUT_TABLES, 'best_model_per_commodity.csv')}")

# ============================================================================
# 3. BUAT FINAL FORECAST (pakai model terbaik per komoditas)
# ============================================================================
final_records = []

fc_map = {
    'SARIMA' : fc_sarima,
    'ETS'    : fc_ets,
    'Prophet': fc_prophet,
}

for _, row in df_cmp.iterrows():
    commodity  = row['Commodity']
    best_model = row['Best_Model']
    fc_source  = fc_map[best_model]

    sub = fc_source[fc_source['commodity'] == commodity].copy()
    sub['best_model'] = best_model
    final_records.append(sub)

df_final = pd.concat(final_records, ignore_index=True)
df_final['date'] = pd.to_datetime(df_final['date'])

# Simpan format long
final_file = os.path.join(OUTPUT_TABLES, "final_forecast_2026_2030.csv")
df_final.to_csv(final_file, index=False)
print(f"[OK] Final forecast (long): {final_file}")

# Simpan format wide (pivot)
fc_wide = df_final.pivot(index='date', columns='commodity', values='forecast')
fc_wide.index.name = 'date'
wide_file = os.path.join(OUTPUT_TABLES, "final_forecast_wide.csv")
fc_wide.to_csv(wide_file)
print(f"[OK] Final forecast (wide): {wide_file}")

# ============================================================================
# 4. TABEL PROYEKSI HARGA KEY MILESTONES
# ============================================================================
print("\n" + "=" * 80)
print("PROYEKSI HARGA  -  KEY MILESTONES (IDR)")
print("=" * 80)

milestones = ['2025-05', '2026-12', '2028-12', '2030-12']
milestone_labels = ['Mei 2025\n(data terakhir)', 'Des 2026', 'Des 2028', 'Des 2030']

milestone_rows = []
for comm in df_cmp['Commodity']:
    row_data = {'Commodity': comm}

    # Ambil nilai historis terakhir
    if comm in df_hist.columns:
        row_data['Mei_2025_hist'] = round(df_hist[comm].loc['2025-05-01':'2025-05-31'].mean(), 0)

    # Ambil dari final forecast
    fc_comm = df_final[df_final['commodity'] == comm].sort_values('date')
    for ms, label in zip(['2026-12', '2028-12', '2030-12'],
                          ['Des_2026', 'Des_2028', 'Des_2030']):
        sub = fc_comm[fc_comm['date'].dt.strftime('%Y-%m') == ms]
        if len(sub) > 0:
            row_data[label] = round(sub['forecast'].values[0], 0)
            row_data[f'{label}_lo'] = round(sub['lower_95'].values[0], 0)
            row_data[f'{label}_hi'] = round(sub['upper_95'].values[0], 0)

    row_data['Best_Model'] = df_cmp[df_cmp['Commodity']==comm]['Best_Model'].values[0]
    milestone_rows.append(row_data)

df_milestones = pd.DataFrame(milestone_rows)

# Hitung % kenaikan 2025->2030
df_milestones['Naik_2030_%'] = (
    (df_milestones['Des_2030'] - df_milestones['Mei_2025_hist'])
    / df_milestones['Mei_2025_hist'] * 100
).round(1)

print(f"\n  {'Commodity':<30} {'Mei2025':>9} {'Des2026':>9} {'Des2028':>9} {'Des2030':>9} {'Naik%':>8} {'Model':>8}")
print(f"  {'-'*80}")
for _, r in df_milestones.iterrows():
    print(f"  {r['Commodity']:<30} "
          f"{r.get('Mei_2025_hist',0):>9,.0f} "
          f"{r.get('Des_2026',0):>9,.0f} "
          f"{r.get('Des_2028',0):>9,.0f} "
          f"{r.get('Des_2030',0):>9,.0f} "
          f"{r.get('Naik_2030_%',0):>7.1f}% "
          f"{r.get('Best_Model',''):>8}")

df_milestones.to_csv(os.path.join(OUTPUT_TABLES, "price_milestones_2026_2030.csv"), index=False)
print(f"\n[OK] Price milestones: {os.path.join(OUTPUT_TABLES, 'price_milestones_2026_2030.csv')}")

# ============================================================================
# 5. VISUALISASI  -  BEST MODEL SUMMARY (Bar + Color-coded Table)
# ============================================================================
print("\n[>] Membuat visualisasi best model summary...")

MODEL_COLORS = {'SARIMA': '#E74C3C', 'ETS': '#8E44AD', 'Prophet': '#27AE60'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (A) Horizontal bar  -  MAPE per komoditas, diwarnai per model
ax = axes[0]
colors = [MODEL_COLORS[m] for m in df_cmp['Best_Model']]
bars = ax.barh(df_cmp['Commodity'], df_cmp['Best_MAPE'],
               color=colors, alpha=0.88, edgecolor='white', height=0.65)
for bar, val in zip(bars, df_cmp['Best_MAPE']):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}%', va='center', fontsize=9)
ax.set_xlabel('MAPE Best Model (%)', fontsize=11)
ax.set_title('MAPE Terbaik per Komoditas\n(model terpilih)', fontsize=12, fontweight='bold')
ax.axvline(10, color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax.text(10.2, -0.7, '10%', fontsize=8, color='gray')
patches = [mpatches.Patch(color=c, label=m) for m, c in MODEL_COLORS.items()]
ax.legend(handles=patches, fontsize=10, loc='lower right')
ax.grid(True, axis='x', alpha=0.3)

# (B) Donut chart  -  model wins
ax2 = axes[1]
win_counts = [wins.get(m, 0) for m in ['SARIMA', 'ETS', 'Prophet']]
win_labels = [f'{m}\n({n} komoditas)' for m, n in zip(['SARIMA','ETS','Prophet'], win_counts)]
win_colors = [MODEL_COLORS[m] for m in ['SARIMA','ETS','Prophet']]
wedges, texts, autotexts = ax2.pie(
    win_counts, labels=win_labels, colors=win_colors,
    autopct='%1.0f%%', startangle=90,
    wedgeprops=dict(width=0.5),   # donut
    textprops={'fontsize': 11}
)
for at in autotexts:
    at.set_fontsize(12)
    at.set_fontweight('bold')
ax2.set_title('Distribusi Model Terbaik\n(berdasarkan MAPE)', fontsize=12, fontweight='bold')

plt.suptitle('Phase 7  -  Model Comparison & Selection', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_FIGURES, "20_best_model_summary.png"),
            dpi=150, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: 20_best_model_summary.png")

# ============================================================================
# 6. VISUALISASI  -  FINAL FORECAST SEMUA KOMODITAS
# ============================================================================
print("[>] Membuat visualisasi final forecast semua komoditas...")

COMMODITIES = list(df_cmp['Commodity'])
n_cols = 2
n_rows = (len(COMMODITIES) + 1) // n_cols
fig2, axes2 = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes2 = axes2.flatten()

for idx, commodity in enumerate(COMMODITIES):
    ax = axes2[idx]
    best_model = df_cmp[df_cmp['Commodity'] == commodity]['Best_Model'].values[0]
    color = MODEL_COLORS[best_model]

    # Historis (zoom 5 tahun terakhir)
    hist = df_hist[commodity].loc['2019-01-01':] if commodity in df_hist.columns else pd.Series()
    if len(hist) > 0:
        ax.plot(hist.index, hist.values, color='#2C3E50',
                linewidth=2, label='Historis', alpha=0.9)

    # Final forecast
    fc_comm = df_final[df_final['commodity'] == commodity].sort_values('date')
    if len(fc_comm) > 0:
        ax.plot(fc_comm['date'], fc_comm['forecast'],
                color=color, linewidth=2.5,
                label=f'Forecast ({best_model})')
        ax.fill_between(
            fc_comm['date'], fc_comm['lower_95'], fc_comm['upper_95'],
            color=color, alpha=0.18, label='CI 95%'
        )

    # Milestone markers
    for ms_date, ms_label in [('2026-12-01','2026'), ('2028-12-01','2028'), ('2030-12-01','2030')]:
        sub = fc_comm[fc_comm['date'] == pd.Timestamp(ms_date)]
        if len(sub) > 0:
            ax.axvline(pd.Timestamp(ms_date), color='gray', linestyle=':', linewidth=0.8, alpha=0.5)
            ax.text(pd.Timestamp(ms_date), ax.get_ylim()[0] if ax.get_ylim()[0] != 0 else 0,
                    ms_label, fontsize=7, color='gray', ha='center')

    ax.set_title(f'{commodity}  [{best_model}]', fontsize=10, fontweight='bold',
                 color=color)
    ax.set_ylabel('Harga (IDR)', fontsize=8)
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True, alpha=0.25)

for j in range(len(COMMODITIES), len(axes2)):
    fig2.delaxes(axes2[j])

plt.suptitle('Final Forecast Harga Pangan Indonesia 2026-2030\n(Model Terbaik per Komoditas)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig2.savefig(os.path.join(OUTPUT_FIGURES, "21_final_forecast_all.png"),
             dpi=150, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: 21_final_forecast_all.png")

# ============================================================================
# 7. VISUALISASI  -  MAPE COMPARISON HEATMAP (komoditas x model)
# ============================================================================
print("[>] Membuat MAPE heatmap 3 model...")

heatmap_data = df_cmp.set_index('Commodity')[['MAPE_SARIMA','MAPE_ETS','MAPE_Prophet']]
heatmap_data.columns = ['SARIMA', 'ETS', 'Prophet']

fig3, ax3 = plt.subplots(figsize=(10, 7))
im = ax3.imshow(heatmap_data.values, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=50)
plt.colorbar(im, ax=ax3, label='MAPE (%)')

ax3.set_xticks(range(3))
ax3.set_xticklabels(['SARIMA (log)', 'ETS (Holt-Winters)', 'Prophet'], fontsize=11)
ax3.set_yticks(range(len(heatmap_data)))
ax3.set_yticklabels(heatmap_data.index, fontsize=10)

for i in range(len(heatmap_data)):
    for j in range(3):
        val = heatmap_data.values[i, j]
        # Bold jika model terbaik untuk komoditas ini
        best = df_cmp.iloc[i]['Best_Model']
        is_best = (['SARIMA','ETS','Prophet'][j] == best)
        weight = 'bold' if is_best else 'normal'
        border = '[ ' if is_best else '  '
        border_e = ' ]' if is_best else '  '
        ax3.text(j, i, f'{border}{val:.1f}%{border_e}',
                 ha='center', va='center', fontsize=9,
                 fontweight=weight,
                 color='white' if val > 30 else 'black')

ax3.set_title('MAPE Comparison Heatmap  -  SARIMA vs ETS vs Prophet\n'
              '(kotak bold = model terbaik per komoditas)',
              fontsize=12, fontweight='bold')
plt.tight_layout()
fig3.savefig(os.path.join(OUTPUT_FIGURES, "22_mape_heatmap.png"),
             dpi=150, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: 22_mape_heatmap.png")

# ============================================================================
# 8. PRINT SUMMARY AKHIR
# ============================================================================
print("\n" + "=" * 80)
print("RINGKASAN PHASE 7  -  FINAL SELECTION")
print("=" * 80)

print("\n--- MODEL TERPILIH PER KOMODITAS ---")
for _, r in df_cmp.iterrows():
    print(f"  {r['Commodity']:<30} -> {r['Best_Model']:<8}  MAPE={r['Best_MAPE']:.2f}%")

print("\n--- PROYEKSI HARGA DESEMBER 2030 (FINAL) ---")
print(f"  {'Commodity':<30} {'Mei 2025':>10} {'Des 2030':>10} {'Kenaikan':>10}  Model")
print(f"  {'-'*68}")
for _, r in df_milestones.sort_values('Naik_2030_%', ascending=False).iterrows():
    naik = r.get('Naik_2030_%', 0)
    icon = 'naik' if naik >= 0 else 'turun'
    print(f"  {r['Commodity']:<30} "
          f"{r.get('Mei_2025_hist',0):>10,.0f} "
          f"{r.get('Des_2030',0):>10,.0f} "
          f"{naik:>+9.1f}%  "
          f"{r.get('Best_Model','')}")

avg_mape = df_cmp['Best_MAPE'].mean()
n_good   = (df_cmp['Best_MAPE'] < 10).sum()
print(f"\n  Rata-rata MAPE terbaik : {avg_mape:.2f}%")
print(f"  Komoditas MAPE < 10%   : {n_good} dari 11")

print("""
[DONE] PHASE 7 COMPLETE
   Files tersimpan:
     - outputs/tables/best_model_per_commodity.csv
     - outputs/tables/final_forecast_2026_2030.csv
     - outputs/tables/final_forecast_wide.csv
     - outputs/tables/price_milestones_2026_2030.csv
     - outputs/figures/20_best_model_summary.png
     - outputs/figures/21_final_forecast_all.png
     - outputs/figures/22_mape_heatmap.png

   Next Step --> Phase 8: Long-term Forecast 2031-2045 (3 Skenario)
""")


## Tahap: 08_longterm_forecast.py

In [ ]:
"""
PHASE 8: LONG-TERM FORECAST 2031-2045 (3 SKENARIO)
Ekstrapolasi dari Phase 7 final forecast menggunakan CAGR + 3 skenario:
  Skenario 1 - Optimis   : Kebijakan ketahanan pangan berhasil (CAGR * 0.5)
  Skenario 2 - Baseline  : Tren saat ini berlanjut (CAGR as-is)
  Skenario 3 - Pesimis   : Tekanan global/iklim (CAGR * 1.5 + 2% floor)
"""

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ============================================================================
# SETUP PATH PORTABLE
# ============================================================================
def _resolve_project_dir():
    """Ambil root proyek; kalau dijalankan dari folder Code, naik ke folder induk."""
    env_dir = os.environ.get("PROJECT_DIR")
    if env_dir:
        return os.path.abspath(env_dir)

    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()

    base_dir = os.path.abspath(base_dir)
    parent_dir = os.path.abspath(os.path.join(base_dir, os.pardir))

    # Struktur proyek: <PROJECT_DIR>/Code/code.ipynb, <PROJECT_DIR>/dataset, <PROJECT_DIR>/outputs
    # Jadi saat notebook/script dijalankan dari folder Code, output tetap ke folder outputs di root proyek.
    if os.path.basename(base_dir).lower() == "code":
        return parent_dir

    if os.path.isdir(os.path.join(base_dir, "dataset")):
        return base_dir
    if os.path.isdir(os.path.join(parent_dir, "dataset")):
        return parent_dir

    return base_dir

PROJECT_DIR = _resolve_project_dir()
DATA  = os.path.join(PROJECT_DIR, "outputs")
FIGS  = os.path.join(DATA, "figures")
TABS  = os.path.join(DATA, "tables")

os.makedirs(DATA, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)
os.makedirs(TABS, exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150

print("=" * 80)
print("PHASE 8: LONG-TERM FORECAST 2031-2045 (3 SKENARIO)")
print("=" * 80)

# ============================================================================
# 1. LOAD DATA
# ============================================================================
df_hist = pd.read_csv(os.path.join(DATA, "national_avg_clean_fixed.csv"),
                      parse_dates=['date'], index_col='date')
df_hist.index = pd.to_datetime(df_hist.index).to_period('M').to_timestamp()

df_final = pd.read_csv(os.path.join(TABS, "final_forecast_2026_2030.csv"))
df_final['date'] = pd.to_datetime(df_final['date'])

df_miles = pd.read_csv(os.path.join(TABS, "price_milestones_2026_2030.csv"))

COMMODITIES = df_miles['Commodity'].tolist()

print(f"\n[OK] Data dimuat  -  {len(COMMODITIES)} komoditas")

# ============================================================================
# 2. HITUNG CAGR BERBASIS HISTORIS JANGKA PANJANG (2011-2026)
# ============================================================================
# PERBAIKAN: Sebelumnya CAGR dihitung dari P_2025->P_2030 (hanya 4-5 tahun
# dari SARIMA forecast). Masalahnya: SARIMA menangkap inflasi pasca-2022
# yang tidak representatif untuk tren 15 tahun ke depan.
#
# Solusi: gunakan CAGR historis 15 tahun (2011->2026) sebagai dasar.
# Ini mencerminkan laju pertumbuhan STRUKTURAL yang lebih stabil.
# Referensi: Inflasi pangan Indonesia BPS ~3-5%/tahun rata-rata jangka panjang.
#
# Metodologi blended:
#   CAGR_final = 0.6 * CAGR_hist15yr + 0.4 * CAGR_forecast4yr
#   - 60% bobot ke historis (stabilitas jangka panjang)
#   - 40% bobot ke forecast (menangkap tren terkini)
#   - Kemudian di-cap agar tidak melampaui batas realistis
# ============================================================================

# Load data historis untuk hitung CAGR 15 tahun
_hist_path = os.path.join(DATA, 'national_avg_clean_fixed.csv')
df_hist_cagr = pd.read_csv(_hist_path, parse_dates=['date'], index_col='date')

cagr_data = {}
for _, row in df_miles.iterrows():
    comm = row['Commodity']
    p_2030 = row.get('Des_2030', np.nan)
    p_2025 = row.get('Mei_2025_hist', np.nan)

    # --- CAGR dari forecast jangka pendek (2025->2030) ---
    if pd.isna(p_2025) or pd.isna(p_2030) or p_2025 <= 0:
        cagr_forecast = 0.03
    else:
        cagr_forecast = (p_2030 / p_2025) ** (1 / 5.58) - 1

    # --- CAGR historis 15 tahun (Jan 2011 -> Jan 2026) ---
    rows_2011 = df_hist_cagr[df_hist_cagr.index.strftime('%Y-%m') == '2011-01']
    rows_2026 = df_hist_cagr[df_hist_cagr.index.strftime('%Y-%m') == '2026-01']
    if comm in df_hist_cagr.columns and len(rows_2011) > 0 and len(rows_2026) > 0:
        p_2011 = rows_2011[comm].values[0]
        p_2026 = rows_2026[comm].values[0]
        if p_2011 > 0 and not np.isnan(p_2011):
            cagr_hist15 = (p_2026 / p_2011) ** (1 / 15) - 1
        else:
            cagr_hist15 = cagr_forecast
    else:
        cagr_hist15 = cagr_forecast

    # --- Blended CAGR: 60% historis + 40% forecast ---
    cagr_blended = 0.60 * cagr_hist15 + 0.40 * cagr_forecast

    # -----------------------------------------------------------------------
    # CAP CAGR: batas atas berdasarkan karakteristik masing-masing komoditas
    # Referensi: inflasi pangan BPS, volatilitas historis, dan kebijakan pangan
    #
    # Catatan metodologis:
    # - Cabai punya volatilitas tinggi (seasonal, supply-shock) namun bukan
    #   tren pertumbuhan struktural -> cap 9%
    # - Pangan pokok (beras, ayam, telur) tumbuh lebih lambat -> cap 7-8%
    # -----------------------------------------------------------------------
    CAGR_CAP = {
        "Chili (bird's eye)": 0.09,   # 9% (volatil tapi non-structural)
        "Chili (red)"       : 0.09,   # 9%
        "Rice"              : 0.07,   # 7% (pangan pokok strategis)
        "Meat (chicken, broiler)": 0.07,
        "Eggs"              : 0.07,
        "Meat (beef)"       : 0.06,
        "Oil (vegetable)"   : 0.06,
        "Sugar"             : 0.05,
    }
    cap = CAGR_CAP.get(comm, 0.07)
    cagr_final = min(cagr_blended, cap)   # tidak melampaui batas realistis
    cagr_final = max(cagr_final, 0.01)    # floor 1%/tahun

    cagr_data[comm] = {
        'cagr'        : cagr_final,
        'cagr_hist15' : cagr_hist15,
        'cagr_forecast': cagr_forecast,
        'p_2030'      : p_2030,
        'p_2025'      : p_2025,
    }

print("\n[INFO] CAGR Long-term per komoditas (blended historis+forecast, dengan cap):")
print(f"  {'Commodity':<30} {'CAGR_Hist15':>12} {'CAGR_FC4yr':>11} {'CAGR_Final':>11}  {'P_2030':>10}")
print("  " + "-" * 80)
for comm, d in cagr_data.items():
    flag = " [CAPPED]" if abs(d['cagr'] - CAGR_CAP.get(comm, 0.07)) < 0.001 else ""
    print(f"  {comm:<30} {d['cagr_hist15']*100:>11.2f}% {d['cagr_forecast']*100:>10.2f}% {d['cagr']*100:>10.2f}%  {d['p_2030']:>10,.0f}{flag}")

# ============================================================================
# 3. DEFINISI SKENARIO
# ============================================================================
# Setiap skenario punya CAGR modifier dan floor tahunan minimum
SCENARIOS = {
    'Optimis': {
        'modifier'   : 0.50,    # CAGR * 0.5  -  kebijakan harga terkendali
        'floor_annual': 0.005,  # min 0.5%/tahun (tidak deflasi ekstrem)
        'color'      : '#27AE60',
        'linestyle'  : '-',
        'desc'       : 'Kebijakan ketahanan pangan efektif, inflasi terkendali',
    },
    'Baseline': {
        'modifier'   : 1.00,    # ikuti tren 2025-2030
        'floor_annual': None,
        'color'      : '#2C3E50',
        'linestyle'  : '--',
        'desc'       : 'Kondisi normal, tren historis berlanjut',
    },
    'Pesimis': {
        'modifier'   : 1.50,    # CAGR * 1.5  -  tekanan global/iklim
        'floor_annual': 0.02,   # min 2%/tahun (harga tidak turun di skenario buruk)
        'color'      : '#E74C3C',
        'linestyle'  : '-.',
        'desc'       : 'Tekanan harga global, perubahan iklim, gangguan rantai pasok',
    },
}

# ============================================================================
# 4. EKSTRAPOLASI BULANAN 2031-2045
# ============================================================================
PROJ_START = pd.Timestamp('2031-01-01')
PROJ_END   = pd.Timestamp('2045-12-01')
proj_dates = pd.date_range(PROJ_START, PROJ_END, freq='MS')
N_MONTHS   = len(proj_dates)

print(f"\n[INFO] Rentang proyeksi: {PROJ_START.strftime('%b %Y')} - {PROJ_END.strftime('%b %Y')}")
print(f"       Jumlah bulan    : {N_MONTHS}")

all_records = []

for scenario_name, sc in SCENARIOS.items():
    for comm, d in cagr_data.items():
        base_cagr    = d['cagr']
        p_start      = d['p_2030']   # harga Des 2030 sebagai starting point

        # Terapkan modifier dan floor
        adj_cagr = base_cagr * sc['modifier']
        if sc['floor_annual'] is not None:
            adj_cagr = max(adj_cagr, sc['floor_annual'])

        # Konversi ke laju bulanan
        monthly_rate = (1 + adj_cagr) ** (1/12) - 1

        # Proyeksi bulan demi bulan
        price = p_start
        for i, dt in enumerate(proj_dates):
            price = price * (1 + monthly_rate)
            # CI melebar seiring waktu: +/-(10% + 2% per tahun dari 2030)
            years_from_2030 = (dt.year - 2030) + (dt.month - 1) / 12
            ci_margin = price * (0.10 + 0.02 * years_from_2030)

            all_records.append({
                'date'     : dt,
                'commodity': comm,
                'scenario' : scenario_name,
                'forecast' : round(price, 2),
                'lower_95' : round(max(price - ci_margin, 0), 2),
                'upper_95' : round(price + ci_margin, 2),
                'cagr_used': round(adj_cagr * 100, 3),
            })

df_lt = pd.DataFrame(all_records)
df_lt['date'] = pd.to_datetime(df_lt['date'])

# Simpan
lt_file = os.path.join(TABS, "longterm_forecast_2031_2045.csv")
df_lt.to_csv(lt_file, index=False)
print(f"\n[OK] Long-term forecast: {lt_file}")
print(f"     Shape: {df_lt.shape}")

# ============================================================================
# 5. TABEL MILESTONES 2031-2045
# ============================================================================
key_years = [2035, 2040, 2045]
mile_rows = []
for comm in COMMODITIES:
    row = {'Commodity': comm, 'P_2030': round(cagr_data[comm]['p_2030'], 0)}
    for sc_name in SCENARIOS:
        sub = df_lt[(df_lt['commodity'] == comm) & (df_lt['scenario'] == sc_name)]
        for yr in key_years:
            yr_data = sub[sub['date'].dt.year == yr]
            if len(yr_data) > 0:
                row[f'{sc_name}_{yr}'] = round(yr_data['forecast'].mean(), 0)
    mile_rows.append(row)

df_lt_miles = pd.DataFrame(mile_rows)
mile_file = os.path.join(TABS, "longterm_milestones_2031_2045.csv")
df_lt_miles.to_csv(mile_file, index=False)
print(f"[OK] Long-term milestones: {mile_file}")

# Print tabel
print("\n" + "=" * 80)
print("PROYEKSI HARGA 2045 (IDR)  -  3 SKENARIO")
print("=" * 80)
print(f"\n  {'Commodity':<28} {'2030':>10} {'Opt_2045':>11} {'Base_2045':>11} {'Pes_2045':>11}")
print(f"  {'-'*73}")
for _, r in df_lt_miles.iterrows():
    p30  = r.get('P_2030', 0)
    opt  = r.get('Optimis_2045', 0)
    base = r.get('Baseline_2045', 0)
    pes  = r.get('Pesimis_2045', 0)
    print(f"  {r['Commodity']:<28} {p30:>10,.0f} {opt:>11,.0f} {base:>11,.0f} {pes:>11,.0f}")

# ============================================================================
# 6. VISUALISASI  -  MAIN PLOT: 3 SKENARIO PER KOMODITAS
# ============================================================================
print("\n[>] Membuat visualisasi 3 skenario per komoditas...")

n_cols = 2
n_rows = (len(COMMODITIES) + 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

# Gabungkan historis + short-term + long-term untuk konteks
for idx, comm in enumerate(COMMODITIES):
    ax = axes[idx]

    # Historis (2019-2025)
    hist = df_hist[comm].loc['2019-01-01':] if comm in df_hist.columns else pd.Series()
    if len(hist) > 0:
        ax.plot(hist.index, hist.values, color='#2C3E50', lw=2,
                label='Historis', alpha=0.9, zorder=5)

    # Short-term (2026-2030) dari Phase 7
    st = df_final[df_final['commodity'] == comm].sort_values('date')
    if len(st) > 0:
        ax.plot(st['date'], st['forecast'], color='#7F8C8D', lw=1.5,
                linestyle=':', label='Forecast 2026-2030', alpha=0.85, zorder=4)

    # Long-term 3 skenario (2031-2045)
    for sc_name, sc in SCENARIOS.items():
        sub = df_lt[(df_lt['commodity'] == comm) & (df_lt['scenario'] == sc_name)].sort_values('date')
        if len(sub) > 0:
            ax.plot(sub['date'], sub['forecast'],
                    color=sc['color'], lw=2.2,
                    linestyle=sc['linestyle'],
                    label=f'Skenario {sc_name}', zorder=3)
            if sc_name == 'Baseline':
                ax.fill_between(sub['date'], sub['lower_95'], sub['upper_95'],
                                color=sc['color'], alpha=0.1)

    # Garis pemisah 2031
    ax.axvline(PROJ_START, color='orange', linestyle='--', lw=1.2, alpha=0.7)
    ax.text(PROJ_START, ax.get_ylim()[1] * 0.02 if ax.get_ylim()[1] > 0 else 0,
            '2031', fontsize=7, color='orange', ha='center')

    # Milestone 2045
    data_2045 = df_lt[(df_lt['commodity'] == comm) & (df_lt['date'].dt.year == 2045)]
    if len(data_2045) > 0:
        for sc_name, sc in SCENARIOS.items():
            val = data_2045[data_2045['scenario'] == sc_name]['forecast'].mean()
            ax.annotate(f'{val:,.0f}',
                        xy=(pd.Timestamp('2045-12-01'), val),
                        fontsize=6.5, color=sc['color'],
                        ha='left', va='center')

    ax.set_title(f'{comm}', fontsize=10, fontweight='bold')
    ax.set_ylabel('Harga (IDR)', fontsize=8)
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(
        lambda x, _: f'{x/1000:.0f}k' if x >= 1000 else f'{x:.0f}'))
    if idx == 0:
        ax.legend(fontsize=6.5, loc='upper left', ncol=2)
    ax.grid(True, alpha=0.2)

for j in range(len(COMMODITIES), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Proyeksi Harga Pangan Indonesia 2031-2045\n'
             '(Skenario Optimis | Baseline | Pesimis)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(FIGS, "23_longterm_three_scenarios.png"),
            dpi=150, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: 23_longterm_three_scenarios.png")

# ============================================================================
# 7. VISUALISASI  -  HEATMAP HARGA 2045
# ============================================================================
print("[>] Membuat heatmap harga 2045...")

hm_data = df_lt_miles.set_index('Commodity')[
    ['Optimis_2045', 'Baseline_2045', 'Pesimis_2045']].copy()
hm_data.columns = ['Optimis', 'Baseline', 'Pesimis']

# Normalisasi per baris (relatif terhadap harga 2030) untuk perbandingan
p2030_vals = df_lt_miles.set_index('Commodity')['P_2030']
hm_ratio   = hm_data.div(p2030_vals, axis=0)

fig2, axes2 = plt.subplots(1, 2, figsize=(15, 7))

# (A) Heatmap nilai absolut IDR
ax = axes2[0]
im = ax.imshow(np.log1p(hm_data.values), aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(3))
ax.set_xticklabels(['Optimis', 'Baseline', 'Pesimis'], fontsize=11)
ax.set_yticks(range(len(hm_data)))
ax.set_yticklabels(hm_data.index, fontsize=9)
for i in range(len(hm_data)):
    for j in range(3):
        ax.text(j, i, f'Rp {hm_data.values[i,j]:,.0f}',
                ha='center', va='center', fontsize=8,
                color='white' if hm_data.values[i,j] > hm_data.values.max()*0.6 else 'black')
ax.set_title('Proyeksi Harga Desember 2045 (IDR)', fontsize=11, fontweight='bold')

# (B) Heatmap rasio vs 2030
ax2 = axes2[1]
im2 = ax2.imshow(hm_ratio.values, aspect='auto', cmap='RdYlGn_r', vmin=0.8, vmax=3.5)
plt.colorbar(im2, ax=ax2, label='Rasio vs harga 2030')
ax2.set_xticks(range(3))
ax2.set_xticklabels(['Optimis', 'Baseline', 'Pesimis'], fontsize=11)
ax2.set_yticks(range(len(hm_ratio)))
ax2.set_yticklabels(hm_ratio.index, fontsize=9)
for i in range(len(hm_ratio)):
    for j in range(3):
        val = hm_ratio.values[i, j]
        ax2.text(j, i, f'{val:.2f}x',
                 ha='center', va='center', fontsize=9, fontweight='bold',
                 color='white' if val > 2.5 else 'black')
ax2.set_title('Kenaikan vs Harga 2030 (rasio)', fontsize=11, fontweight='bold')

plt.suptitle('Proyeksi Harga Pangan 2045  -  Heatmap\n'
             '(Menuju Indonesia Emas 2045)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fig2.savefig(os.path.join(FIGS, "24_longterm_heatmap_2045.png"),
             dpi=150, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: 24_longterm_heatmap_2045.png")

# ============================================================================
# 8. VISUALISASI  -  COMPOSITE FOOD PRICE INDEX (CFPI)
# ============================================================================
print("[>] Membuat Composite Food Price Index...")

# Bobot Composite Food Price Index setelah penghapusan 3 komoditas.
# Normalisasi proporsional dari bobot asli 8 komoditas tersisa (total approx. 1.00).
WEIGHTS = {
    'Rice'                   : 0.238,
    'Meat (beef)'            : 0.143,
    'Meat (chicken, broiler)': 0.143,
    'Eggs'                   : 0.119,
    'Oil (vegetable)'        : 0.119,
    'Sugar'                  : 0.095,
    "Chili (bird's eye)"     : 0.071,
    'Chili (red)'            : 0.071,
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 0.002, "Total bobot CFPI harus approx. 1.00"

# Base index = rata-rata historis 2007-2025
base_vals = {comm: df_hist[comm].mean() for comm in WEIGHTS if comm in df_hist.columns}

fig3, ax3 = plt.subplots(figsize=(14, 6))

for sc_name, sc in SCENARIOS.items():
    cfpi_vals = []
    cfpi_dates = []
    for dt in proj_dates:
        idx_val = 0.0
        for comm, w in WEIGHTS.items():
            sub = df_lt[(df_lt['commodity'] == comm) &
                        (df_lt['scenario'] == sc_name) &
                        (df_lt['date'] == dt)]
            if len(sub) > 0 and comm in base_vals and base_vals[comm] > 0:
                idx_val += w * (sub['forecast'].values[0] / base_vals[comm]) * 100
        cfpi_vals.append(idx_val)
        cfpi_dates.append(dt)

    ax3.plot(cfpi_dates, cfpi_vals,
             color=sc['color'], lw=2.5,
             linestyle=sc['linestyle'],
             label=f"Skenario {sc_name}",
             zorder=3)

# Historis CFPI
hist_cfpi = []
for dt in df_hist.index:
    iv = 0.0
    for comm, w in WEIGHTS.items():
        if comm in df_hist.columns and comm in base_vals and base_vals[comm] > 0:
            iv += w * (df_hist.loc[dt, comm] / base_vals[comm]) * 100
    hist_cfpi.append(iv)

ax3.plot(df_hist.index, hist_cfpi, color='#2C3E50', lw=2, label='Historis', zorder=5)

ax3.axhline(100, color='gray', linestyle=':', lw=1.2, alpha=0.7, label='Basis (avg 2007-2025)')
ax3.axvline(PROJ_START, color='orange', linestyle='--', lw=1.2, alpha=0.7)
ax3.text(PROJ_START, ax3.get_ylim()[0] if ax3.get_ylim() else 80,
         'Proyeksi', fontsize=8, color='orange', rotation=90, va='bottom')

ax3.set_title('Composite Food Price Index (CFPI)\nIndonesia 2007-2045',
              fontsize=13, fontweight='bold')
ax3.set_ylabel('Index (Basis = rata-rata 2007-2025 = 100)', fontsize=10)
ax3.set_xlabel('Tahun', fontsize=10)
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.25)

plt.tight_layout()
fig3.savefig(os.path.join(FIGS, "25_composite_food_price_index.png"),
             dpi=150, bbox_inches='tight')
plt.close()
print(f"   [OK] Tersimpan: 25_composite_food_price_index.png")

# ============================================================================
# 9. SUMMARY AKHIR
# ============================================================================
print("\n" + "=" * 80)
print("RINGKASAN PHASE 8  -  PROYEKSI 2031-2045")
print("=" * 80)

print(f"""
SKENARIO:
  Optimis   -  CAGR 2025-2030 x0.5, min +0.5%/thn
             {SCENARIOS['Optimis']['desc']}
  Baseline  -  CAGR 2025-2030 x1.0 (tren berlanjut)
             {SCENARIOS['Baseline']['desc']}
  Pesimis   -  CAGR 2025-2030 x1.5, min +2%/thn
             {SCENARIOS['Pesimis']['desc']}
""")

# CAGR implied 2030-2045 rata-rata
for sc_name in SCENARIOS:
    sub_2045 = df_lt[(df_lt['date'].dt.year == 2045) & (df_lt['scenario'] == sc_name)]
    sub_2030_anchor = pd.DataFrame([{'commodity': c, 'price_2030': cagr_data[c]['p_2030']}
                                     for c in COMMODITIES])
    merged = sub_2045[['commodity','forecast']].merge(sub_2030_anchor, on='commodity')
    n = 15  # 2030 -> 2045
    merged['cagr_30_45'] = ((merged['forecast'] / merged['price_2030']) ** (1/n) - 1) * 100
    avg_c = merged['cagr_30_45'].mean()
    print(f"  Rata-rata CAGR 2030-2045 ({sc_name:<10}): {avg_c:.2f}%/tahun")

print("""
[DONE] PHASE 8 COMPLETE
   Files tersimpan:
     - outputs/tables/longterm_forecast_2031_2045.csv
     - outputs/tables/longterm_milestones_2031_2045.csv
     - outputs/figures/23_longterm_three_scenarios.png
     - outputs/figures/24_longterm_heatmap_2045.png
     - outputs/figures/25_composite_food_price_index.png

   Next Step --> Phase 9: Laporan & Visualisasi Final
""")


## Tahap: 09_final_report.py

In [ ]:
"""
PHASE 9: VISUALISASI FINAL & RINGKASAN UNTUK ESAI SEC
Dashboard, milestone chart, tabel ringkasan, dan insight ketahanan pangan.
"""
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ============================================================================
# SETUP PATH PORTABLE
# ============================================================================
def _resolve_project_dir():
    """Ambil root proyek; kalau dijalankan dari folder Code, naik ke folder induk."""
    env_dir = os.environ.get("PROJECT_DIR")
    if env_dir:
        return os.path.abspath(env_dir)

    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()

    base_dir = os.path.abspath(base_dir)
    parent_dir = os.path.abspath(os.path.join(base_dir, os.pardir))

    # Struktur proyek: <PROJECT_DIR>/Code/code.ipynb, <PROJECT_DIR>/dataset, <PROJECT_DIR>/outputs
    # Jadi saat notebook/script dijalankan dari folder Code, output tetap ke folder outputs di root proyek.
    if os.path.basename(base_dir).lower() == "code":
        return parent_dir

    if os.path.isdir(os.path.join(base_dir, "dataset")):
        return base_dir
    if os.path.isdir(os.path.join(parent_dir, "dataset")):
        return parent_dir

    return base_dir

PROJECT_DIR = _resolve_project_dir()
DATA = os.path.join(PROJECT_DIR, "outputs")
FIGS = os.path.join(DATA, "figures")
TABS = os.path.join(DATA, "tables")

os.makedirs(DATA, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)
os.makedirs(TABS, exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'DejaVu Sans'

print("=" * 80)
print("PHASE 9: VISUALISASI FINAL & RINGKASAN ESAI SEC")
print("=" * 80)

# ============================================================================
# LOAD DATA
# ============================================================================
df_hist   = pd.read_csv(os.path.join(DATA, "national_avg_clean_fixed.csv"),
                         parse_dates=['date'], index_col='date')
df_hist.index = pd.to_datetime(df_hist.index).to_period('M').to_timestamp()

df_final  = pd.read_csv(os.path.join(TABS, "final_forecast_2026_2030.csv"))
df_final['date'] = pd.to_datetime(df_final['date'])

df_lt     = pd.read_csv(os.path.join(TABS, "longterm_forecast_2031_2045.csv"))
df_lt['date'] = pd.to_datetime(df_lt['date'])

df_cmp    = pd.read_csv(os.path.join(TABS, "best_model_per_commodity.csv"))
df_miles  = pd.read_csv(os.path.join(TABS, "price_milestones_2026_2030.csv"))
df_lt_ms  = pd.read_csv(os.path.join(TABS, "longterm_milestones_2031_2045.csv"))

COMMODITIES = list(df_hist.columns)

SC_COLORS = {'Optimis': '#27AE60', 'Baseline': '#2C3E50', 'Pesimis': '#E74C3C'}
MODEL_COLORS = {'SARIMA': '#E74C3C', 'ETS': '#8E44AD', 'Prophet': '#27AE60'}

print("[OK] Semua data dimuat.\n")

# ============================================================================
# 1. DASHBOARD HISTORIS (2007-2025)  -  11 komoditas panel
# ============================================================================
print("[>] Fig 26: Dashboard tren historis...")

n_cols = 3
n_rows = (len(COMMODITIES) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3.5))
axes = axes.flatten()

for idx, comm in enumerate(COMMODITIES):
    ax = axes[idx]
    s = df_hist[comm].dropna()
    ax.plot(s.index, s.values, color='#2980B9', lw=1.8, alpha=0.9)

    # Rolling mean 12 bulan
    rm = s.rolling(12).mean()
    ax.plot(rm.index, rm.values, color='#E74C3C', lw=1.4, linestyle='--',
            alpha=0.85, label='MA-12')

    # Shade Ramadan spike bulan
    for yr in range(2007, 2026):
        ax.axvspan(pd.Timestamp(f'{yr}-04-01'), pd.Timestamp(f'{yr}-07-01'),
                   alpha=0.04, color='orange')

    ax.set_title(comm, fontsize=9, fontweight='bold', pad=3)
    ax.set_ylabel('Rp', fontsize=7)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(
        lambda x, _: f'{x/1000:.0f}k' if x >= 1000 else f'{x:.0f}'))
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.2)

for j in range(len(COMMODITIES), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Tren Historis Harga Pangan Nasional Indonesia (2007-2025)\n'
             '(area oranye = periode Ramadan/Lebaran, garis merah = MA-12)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(FIGS, "26_dashboard_historis.png"), dpi=150, bbox_inches='tight')
plt.close()
print("   [OK] 26_dashboard_historis.png")

# ============================================================================
# 2. MILESTONE CHART  -  Harga 2025, 2030, 2035, 2040, 2045
# ============================================================================
print("[>] Fig 27: Milestone chart harga 2025-2045...")

def get_price(comm, yr, sc=None):
    """Ambil harga rata-rata suatu tahun dari long-term atau short-term."""
    if yr <= 2030:
        sub = df_final[(df_final['commodity'] == comm) &
                       (df_final['date'].dt.year == yr)]
        return sub['forecast'].mean() if len(sub) > 0 else np.nan
    else:
        sub = df_lt[(df_lt['commodity'] == comm) &
                    (df_lt['scenario'] == sc) &
                    (df_lt['date'].dt.year == yr)]
        return sub['forecast'].mean() if len(sub) > 0 else np.nan

MILESTONES = [2025, 2030, 2035, 2040, 2045]
KEY_COMMS  = ['Rice', 'Meat (beef)', "Chili (bird's eye)", 'Eggs',
              'Sugar', 'Oil (vegetable)', 'Meat (chicken, broiler)']

fig2, axes2 = plt.subplots(len(KEY_COMMS), 1, figsize=(14, len(KEY_COMMS) * 2.2),
                            sharex=False)

for i, comm in enumerate(KEY_COMMS):
    ax = axes2[i]

    # 2025 historis
    p2025 = df_hist[comm].loc['2025-01-01':'2025-05-31'].mean()

    # 2030 dari final forecast
    p2030 = get_price(comm, 2030)

    for sc_name, sc_color in SC_COLORS.items():
        x_vals, y_vals = [2025, 2030], [p2025, p2030]
        for yr in [2035, 2040, 2045]:
            p = get_price(comm, yr, sc=sc_name)
            x_vals.append(yr)
            y_vals.append(p)

        ax.plot(x_vals, y_vals, 'o-', color=sc_color, lw=2,
                ms=6, label=sc_name if i == 0 else None,
                linestyle='-' if sc_name == 'Baseline' else
                          '--' if sc_name == 'Optimis' else '-.')

    ax.set_title(comm, fontsize=9, fontweight='bold', loc='left', pad=2)
    ax.set_ylabel('IDR', fontsize=7)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(
        lambda x, _: f'{x/1000:.0f}k' if x >= 1000 else f'{x:.0f}'))
    ax.set_xticks(MILESTONES)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.2, axis='y')

handles = [mpatches.Patch(color=c, label=s) for s, c in SC_COLORS.items()]
axes2[0].legend(handles=handles, fontsize=9, loc='upper left', ncol=3)
plt.suptitle('Proyeksi Harga Komoditas Utama 2025-2045\n(3 Skenario)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
fig2.savefig(os.path.join(FIGS, "27_milestone_chart_2025_2045.png"),
             dpi=150, bbox_inches='tight')
plt.close()
print("   [OK] 27_milestone_chart_2025_2045.png")

# ============================================================================
# 3. FAN CHART  -  Rice, Daging, Cabai (highlight untuk esai)
# ============================================================================
print("[>] Fig 28: Fan chart 3 komoditas fokus...")

FOCUS = ['Rice', 'Meat (beef)', "Chili (bird's eye)"]
fig3, axes3 = plt.subplots(1, 3, figsize=(18, 5))

for i, comm in enumerate(FOCUS):
    ax = axes3[i]

    # Historis
    hist = df_hist[comm].loc['2015-01-01':]
    ax.plot(hist.index, hist.values, color='#2C3E50', lw=2, label='Historis')

    # Short-term forecast
    st = df_final[df_final['commodity'] == comm].sort_values('date')
    ax.plot(st['date'], st['forecast'], color='#7F8C8D', lw=1.5,
            linestyle=':', label='Short-term')
    ax.fill_between(st['date'], st['lower_95'], st['upper_95'],
                    color='#7F8C8D', alpha=0.15)

    # Long-term 3 skenario  -  fan
    for sc_name, sc_color in SC_COLORS.items():
        sub = df_lt[(df_lt['commodity'] == comm) &
                    (df_lt['scenario'] == sc_name)].sort_values('date')
        ax.plot(sub['date'], sub['forecast'], color=sc_color, lw=2,
                linestyle='-' if sc_name == 'Baseline' else
                          '--' if sc_name == 'Optimis' else '-.',
                label=f'Skenario {sc_name}')
        if sc_name in ('Optimis', 'Pesimis'):
            # Isi area antara optimis dan pesimis
            pass

    # Isi fan antara optimis dan pesimis
    opt = df_lt[(df_lt['commodity']==comm) & (df_lt['scenario']=='Optimis')].sort_values('date')
    pes = df_lt[(df_lt['commodity']==comm) & (df_lt['scenario']=='Pesimis')].sort_values('date')
    if len(opt) == len(pes):
        ax.fill_between(opt['date'], opt['forecast'], pes['forecast'],
                        color='gray', alpha=0.12, label='Range Skenario')

    ax.axvline(pd.Timestamp('2031-01-01'), color='orange',
               linestyle='--', lw=1, alpha=0.7)

    ax.set_title(comm, fontsize=11, fontweight='bold')
    ax.set_ylabel('Harga (IDR)', fontsize=9)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(
        lambda x, _: f'{x/1000:.0f}k' if x >= 1000 else f'{x:.0f}'))
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.legend(fontsize=7.5, loc='upper left')
    ax.grid(True, alpha=0.2)

plt.suptitle('Fan Chart Proyeksi Harga Pangan Pokok 2015-2045\n'
             'Rice | Daging Sapi | Cabai Rawit',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fig3.savefig(os.path.join(FIGS, "28_fan_chart_focus.png"),
             dpi=150, bbox_inches='tight')
plt.close()
print("   [OK] 28_fan_chart_focus.png")

# ============================================================================
# 4. TABEL RINGKASAN ESAI  -  Harga 2025 vs 2030 vs 2045
# ============================================================================
print("[>] Fig 29: Tabel ringkasan harga esai...")

rows = []
for _, r in df_miles.iterrows():
    comm = r['Commodity']
    p25  = r.get('Mei_2025_hist', 0)
    p30  = r.get('Des_2030', 0)
    best = r.get('Best_Model', '')

    opt45  = df_lt_ms[df_lt_ms['Commodity'] == comm].get('Optimis_2045', pd.Series([np.nan])).values
    base45 = df_lt_ms[df_lt_ms['Commodity'] == comm].get('Baseline_2045', pd.Series([np.nan])).values
    pes45  = df_lt_ms[df_lt_ms['Commodity'] == comm].get('Pesimis_2045', pd.Series([np.nan])).values

    opt45  = opt45[0]  if len(opt45) > 0 else np.nan
    base45 = base45[0] if len(base45) > 0 else np.nan
    pes45  = pes45[0]  if len(pes45) > 0 else np.nan

    naik_30  = (p30 - p25) / p25 * 100 if p25 > 0 else 0
    naik_45b = (base45 - p25) / p25 * 100 if p25 > 0 and not np.isnan(base45) else 0

    rows.append({
        'Komoditas'      : comm,
        'Mei 2025'       : f"Rp {p25:,.0f}",
        'Des 2030'       : f"Rp {p30:,.0f}",
        'Kenaikan 30'    : f"{naik_30:+.1f}%",
        '2045 Optimis'   : f"Rp {opt45:,.0f}" if not np.isnan(opt45) else '-',
        '2045 Baseline'  : f"Rp {base45:,.0f}" if not np.isnan(base45) else '-',
        '2045 Pesimis'   : f"Rp {pes45:,.0f}" if not np.isnan(pes45) else '-',
        'Kenaikan 45'    : f"{naik_45b:+.1f}%",
        'Model'          : best,
    })

df_summary = pd.DataFrame(rows)
df_summary.to_csv(os.path.join(TABS, "ringkasan_esai_harga_2025_2045.csv"), index=False)
print(f"   [OK] ringkasan_esai_harga_2025_2045.csv")

# Visualisasi tabel sebagai gambar
fig4, ax4 = plt.subplots(figsize=(20, 5))
ax4.axis('off')
cols = ['Komoditas', 'Mei 2025', 'Des 2030', 'Kenaikan 30',
        '2045 Baseline', '2045 Pesimis', 'Kenaikan 45', 'Model']
tbl_data = df_summary[cols].values.tolist()
tbl = ax4.table(
    cellText=tbl_data,
    colLabels=cols,
    loc='center', cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1, 1.8)

# Warna header
for j in range(len(cols)):
    tbl[0, j].set_facecolor('#2C3E50')
    tbl[0, j].set_text_props(color='white', fontweight='bold')

# Warna baris bergantian
for i in range(1, len(tbl_data) + 1):
    for j in range(len(cols)):
        tbl[i, j].set_facecolor('#EBF5FB' if i % 2 == 0 else 'white')

plt.title('Ringkasan Proyeksi Harga Pangan Nasional Indonesia 2025-2045\n'
          'Sumber: WFP HDX + Model Forecasting (SARIMA/ETS/Prophet)',
          fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
fig4.savefig(os.path.join(FIGS, "29_tabel_ringkasan_esai.png"),
             dpi=150, bbox_inches='tight')
plt.close()
print("   [OK] 29_tabel_ringkasan_esai.png")

# ============================================================================
# 5. INSIGHT UNTUK ESAI  -  Print ringkasan narasi
# ============================================================================
print("\n" + "=" * 80)
print("INSIGHT KUNCI UNTUK ESAI SEC SATRIA DATA 2026")
print("=" * 80)

# Volatilitas tertinggi
vol_rank = {c: df_hist[c].std()/df_hist[c].mean()*100 for c in COMMODITIES}
vol_sorted = sorted(vol_rank.items(), key=lambda x: x[1], reverse=True)

print("\n--- 1. VOLATILITAS HISTORIS (2007-2025) ---")
for comm, cv in vol_sorted:
    bar = '=' * int(cv / 3)
    print(f"  {comm:<30} CV={cv:5.1f}%  {bar}")

# Inflasi historis
print("\n--- 2. INFLASI KUMULATIF 2007-2025 ---")
inflation_rank = []
for comm in COMMODITIES:
    s = df_hist[comm].dropna()
    if len(s) > 0:
        pct = (s.iloc[-1] - s.iloc[0]) / s.iloc[0] * 100
        cagr = ((s.iloc[-1] / s.iloc[0]) ** (1 / 18) - 1) * 100
        inflation_rank.append((comm, pct, cagr))
        print(f"  {comm:<30} total={pct:+7.1f}%  CAGR={cagr:.2f}%/thn")
inflation_sorted = sorted(inflation_rank, key=lambda x: x[1], reverse=True)

# Proyeksi 2045 baseline
print("\n--- 3. PROYEKSI HARGA DESEMBER 2045 (BASELINE) ---")
for _, r in df_lt_ms.iterrows():
    p25  = df_miles[df_miles['Commodity'] == r['Commodity']]['Mei_2025_hist'].values
    p25  = p25[0] if len(p25) > 0 else 0
    b45  = r.get('Baseline_2045', 0)
    mult = b45 / p25 if p25 > 0 else 0
    print(f"  {r['Commodity']:<30} Rp {b45:>10,.0f}  ({mult:.2f}x lipat dari 2025)")

# Insight dinamis: semua angka diambil langsung dari tabel/grafik yang sedang dipakai
print("\n--- 4. KESIMPULAN KETAHANAN PANGAN ---")
_top_vol_comm, _top_vol_cv = vol_sorted[0]
_top_inf_comm, _top_inf_pct, _top_inf_cagr = inflation_sorted[0]
_rice_row = df_lt_ms[df_lt_ms['Commodity'] == 'Rice']
_threat = df_lt_ms.loc[df_lt_ms['Pesimis_2045'].idxmax()]

if len(_rice_row) > 0:
    _rice = _rice_row.iloc[0]
    _rice_text = (
        f"  c) Harga BERAS 2045:\n"
        f"     - Optimis : sekitar Rp {_rice['Optimis_2045']:,.0f}/kg\n"
        f"     - Baseline: sekitar Rp {_rice['Baseline_2045']:,.0f}/kg\n"
        f"     - Pesimis : sekitar Rp {_rice['Pesimis_2045']:,.0f}/kg"
    )
else:
    _rice_text = "  c) Harga BERAS 2045: tidak tersedia di tabel longterm_milestones_2031_2045.csv"

print(
    f"  a) Komoditas PALING VOLATIL: {_top_vol_comm} "
    f"(CV={_top_vol_cv:.1f}%)  -  perlu kebijakan stabilisasi dan buffer stok.\n"
    f"  b) Inflasi TERTINGGI 2007-2025: {_top_inf_comm} "
    f"(total={_top_inf_pct:+.1f}%, CAGR={_top_inf_cagr:.2f}%/thn).\n"
    f"{_rice_text}\n"
    f"  d) ANCAMAN TERBESAR 2045: {_threat['Commodity']} "
    f"di skenario pesimis, sekitar Rp {_threat['Pesimis_2045']:,.0f}.\n"
    f"  e) REKOMENDASI KEBIJAKAN:\n"
    f"     - Stabilisasi harga {_top_vol_comm} lewat cold storage, distribusi antardaerah, dan asuransi pertanian.\n"
    f"     - Prioritaskan intervensi pada {_top_inf_comm} karena inflasi historisnya paling tinggi.\n"
    f"     - Diversifikasi protein dan bahan pangan untuk mengurangi tekanan pada komoditas mahal.\n"
    f"     - Perkuat cadangan pangan nasional untuk komoditas pokok seperti beras."
)

print("=" * 80)
print("[DONE] PHASE 9 COMPLETE")
print("[DONE] PHASE 9 COMPLETE")
print("=" * 80)
print("""
   Files tersimpan:
     - outputs/figures/26_dashboard_historis.png
     - outputs/figures/27_milestone_chart_2025_2045.png
     - outputs/figures/28_fan_chart_focus.png
     - outputs/figures/29_tabel_ringkasan_esai.png
     - outputs/tables/ringkasan_esai_harga_2025_2045.csv

   PIPELINE SELESAI: Phase 0 - 9 sudah lengkap!
   Siap untuk penulisan esai SEC SATRIA DATA 2026.
""")
